In [ ]:
# pypsa_analysis_utils.py

import pypsa
import pandas as pd
import numpy as np
import logging
from typing import Union, Optional, Tuple, Dict, List, Any
from collections import OrderedDict # For consistent JSON output order if needed

# Logging configuration
# Configures basic logging for the module.
# Level INFO means informational messages and above (WARNING, ERROR, CRITICAL) will be logged.
# Format includes timestamp, log level, module name, line number, and the message.
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - [%(module)s:%(lineno)d] - %(message)s'
)
logger = logging.getLogger(__name__) # Get a logger instance for this module

# Default color palette for consistent and recognizable chart colors.
# Keys are typically carrier names (lowercase for broader matching) or component types.
DEFAULT_COLORS = {
    'Coal': '#000000', 'coal': '#000000',
    'Lignite': '#4B4B4B', 'lignite': '#4B4B4B',
    'Nuclear': '#800080', 'nuclear': '#800080',
    'Hydro': '#0073CF', 'hydro': '#0073CF', 
    'Hydro RoR': '#3399FF', 'ror': '#3399FF', 
    'Hydro Storage': '#005293', 
    'Solar': '#FFD700', 'solar': '#FFD700', 'pv': '#FFD700',
    'Wind': '#ADD8E6', 'wind': '#ADD8E6', 'onwind': '#ADD8E6', 'offwind': '#ADD8E6',
    'LFO': '#FF4500', 'lfo': '#FF4500', 'Oil': '#FF4500', 'oil': '#FF4500',
    'Co-Gen': '#228B22', 'co-gen': '#228B22', 'biomass': '#228B22',
    'PSP': '#3399FF', 'psp': '#3399FF', 
    'Battery Storage': '#005B5B', 'battery': '#005B5B',
    'Planned Battery Storage': '#66B2B2', 'planned battery': '#66B2B2',
    'Planned PSP': '#B0C4DE', 'planned psp': '#B0C4DE',
    'Storage': '#B0C4DE', 
    'H2 Storage': '#AFEEEE', 'hydrogen': '#AFEEEE', 'h2': '#AFEEEE', 'H2': '#AFEEEE',
    'Load': '#FF6347', 
    'Transmission': '#808080', 'Line': '#808080', 'Link': '#A9A9A9',
    'Losses': '#DC143C', 
    'Other': '#D3D3D3', 
    'Curtailment': '#FF00FF', 
    'Excess': '#FF00FF',
    'Storage Charge': '#FFA500', 
    'Storage Discharge': '#90EE90', 
    'Store Charge': '#FFC0CB', 
    'Store Discharge': '#87CEEB', 
}

# Plotly's default qualitative color cycle, used as a fallback for unmapped carriers.
PLOTLY_COLOR_CYCLE = [
    "#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A",
    "#19D3F3", "#FF6692", "#B6E880", "#FF97FF", "#FECB52"
]


# --- Core Utility Functions ---

def safe_get_snapshots(n: pypsa.Network) -> Union[pd.DatetimeIndex, pd.MultiIndex, pd.Index]:
    """
    Safely retrieves the network snapshots.
    Returns an empty pd.Index if snapshots attribute doesn't exist or is None.

    Args:
        n (pypsa.Network): The PyPSA network object.

    Returns:
        Union[pd.DatetimeIndex, pd.MultiIndex, pd.Index]: Network snapshots or an empty Index.
    """
    return n.snapshots if hasattr(n, 'snapshots') and n.snapshots is not None else pd.Index([])

def get_time_index(index: Union[pd.DatetimeIndex, pd.MultiIndex, pd.Index, None]) -> Optional[pd.DatetimeIndex]:
    """
    Extracts or converts the time component of a pandas Index to a pd.DatetimeIndex.
    This is crucial for time-series operations like resampling.

    Args:
        index: The pandas Index object (can be DatetimeIndex, MultiIndex, or general Index).

    Returns:
        Optional[pd.DatetimeIndex]: A DatetimeIndex if conversion is successful, otherwise None.
    """
    if index is None or index.empty:
        logger.debug("get_time_index: Received None or empty index.")
        return None
    if isinstance(index, pd.DatetimeIndex):
        return index
    if isinstance(index, pd.MultiIndex):
        # Assume the last level of a MultiIndex is the time component.
        time_level_values = index.get_level_values(-1)
        if pd.api.types.is_datetime64_any_dtype(time_level_values.dtype):
            return pd.DatetimeIndex(time_level_values)
        else: 
            try: # Attempt conversion if not already datetime
                return pd.to_datetime(time_level_values)
            except (ValueError, TypeError, OverflowError) as e:
                logger.debug(f"Could not convert MultiIndex level (type: {time_level_values.dtype}) to DatetimeIndex: {e}")
                return None
    # For general pd.Index or other types, attempt conversion.
    try:
        return pd.to_datetime(index)
    except (ValueError, TypeError, OverflowError) as e:
        logger.debug(f"Cannot convert index of type {type(index)} to DatetimeIndex: {e}")
        return None

def get_period_index(index: Union[pd.DatetimeIndex, pd.MultiIndex, pd.Index, None]) -> Optional[Union[pd.Index, pd.Series]]:
    """
    Extracts the period component from a pandas Index.
    For a MultiIndex, assumes the first level is the period.
    For a DatetimeIndex, defaults to extracting the year as the period.

    Args:
        index: The pandas Index object.

    Returns:
        Optional[Union[pd.Index, pd.Series]]: Period identifiers or None if not applicable.
    """
    if index is None or index.empty:
        logger.debug("get_period_index: Received None or empty index.")
        return None
    if isinstance(index, pd.MultiIndex) and index.nlevels > 0:
        return index.get_level_values(0) 
    elif isinstance(index, pd.DatetimeIndex):
        return pd.Series(index.year, index=index) # Default to year for single DatetimeIndex
    logger.debug(f"Cannot determine period index from index type {type(index)}. Returning None.")
    return None

def get_snapshot_weights(n: pypsa.Network, snapshots_idx: Union[pd.DatetimeIndex, pd.MultiIndex, pd.Index]) -> pd.Series:
    """
    Retrieves snapshot weights (typically representing duration in hours) from the network,
    aligned with the provided `snapshots_idx`. Defaults to 1.0 if weights are not found.

    Args:
        n (pypsa.Network): The PyPSA network object.
        snapshots_idx: The Index of snapshots for which weights are required.

    Returns:
        pd.Series: Snapshot weights aligned to `snapshots_idx`.
    """
    if snapshots_idx.empty:
        return pd.Series(dtype=float, index=snapshots_idx) 
    
    if hasattr(n, 'snapshot_weightings') and \
       not n.snapshot_weightings.empty and \
       'objective' in n.snapshot_weightings.columns:
        
        weights = n.snapshot_weightings.objective
        common_index = snapshots_idx.intersection(weights.index)
        if not common_index.empty:
            return weights.reindex(common_index).reindex(snapshots_idx).fillna(1.0)
        else: 
            logger.debug("No common index between provided snapshots and network's snapshot_weightings. Defaulting weights to 1.0 for given snapshots.")
    else:
        logger.debug("snapshot_weightings.objective not found or empty. Defaulting weights to 1.0 for given snapshots.")
    return pd.Series(1.0, index=snapshots_idx) 

def get_effective_snapshots(n: pypsa.Network, _snapshots_slice: Optional[Union[pd.DatetimeIndex, pd.MultiIndex, pd.Index]] = None) -> Union[pd.DatetimeIndex, pd.MultiIndex, pd.Index]:
    """
    Determines the effective set of snapshots to use for subsequent calculations.
    If `_snapshots_slice` is provided and valid, it's used. Otherwise, all network snapshots are used.

    Args:
        n (pypsa.Network): The PyPSA network object.
        _snapshots_slice: An optional pre-filtered slice of snapshots.

    Returns:
        Union[pd.DatetimeIndex, pd.MultiIndex, pd.Index]: The effective snapshots.
    """
    if _snapshots_slice is not None: 
        if _snapshots_slice.empty:
            logger.info("get_effective_snapshots: Received an explicitly empty _snapshots_slice.")
            return pd.Index([]) 
        return _snapshots_slice 
    else: 
        return safe_get_snapshots(n)

def get_carrier_map(comp_df: pd.DataFrame, carriers_df_optional: Optional[pd.DataFrame], default_carrier_name: Optional[str] = None) -> Optional[pd.Series]:
    """
    Generates a mapping Series from component names to their 'nice_name' carrier representation.
    Uses 'nice_name' from `carriers_df` if available, otherwise original carrier name.
    If 'carrier' column is missing, can use `default_carrier_name`.
    Ensures the returned Series contains string values for reliable grouping.

    Args:
        comp_df (pd.DataFrame): The component DataFrame (e.g., n.generators).
        carriers_df_optional (Optional[pd.DataFrame]): The network's carriers DataFrame (n.carriers).
        default_carrier_name (Optional[str]): A default name if 'carrier' column is absent.

    Returns:
        Optional[pd.Series]: A Series mapping component index to displayable carrier names (strings).
    """
    if 'carrier' not in comp_df.columns and default_carrier_name is None:
        logger.debug(f"Component DataFrame (index: {list(comp_df.index)[:3]}...) lacks 'carrier' column and no default name provided.")
        return None
    
    carrier_map_series = comp_df.get('carrier', pd.Series(default_carrier_name, index=comp_df.index))
    
    carriers_df = carriers_df_optional
    if not isinstance(carriers_df, pd.DataFrame) or carriers_df.empty:
        unique_carriers_in_series = carrier_map_series.dropna().unique()
        carriers_df = pd.DataFrame(index=unique_carriers_in_series)

    if 'nice_name' not in carriers_df.columns: 
        carriers_df = carriers_df.copy() 
        carriers_df['nice_name'] = carriers_df.index 

    nice_name_map = carriers_df['nice_name'].dropna() 
    
    mapped_names = carrier_map_series.map(nice_name_map).fillna(carrier_map_series)
    if default_carrier_name:
         mapped_names = mapped_names.fillna(default_carrier_name)
    
    return mapped_names.astype(str) 


# --- Main Data Extraction Functions ---

def get_dispatch_data(_n: pypsa.Network, 
                      _snapshots_slice: Optional[Union[pd.DatetimeIndex, pd.MultiIndex, pd.Index]] = None,
                      resolution: str = "1H") -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.DataFrame]:
    """
    Extracts and aggregates dispatch data for generators, load, and storage components (StorageUnits, Stores).
    Data is aligned with `effective_snapshots` and can be resampled.

    Args:
        _n (pypsa.Network): The PyPSA network object.
        _snapshots_slice: Optional pre-filtered snapshots to analyze.
        resolution (str): Target time resolution for resampling (e.g., "1H", "3H", "1D").

    Returns:
        Tuple containing:
            - gen_dispatch_agg (pd.DataFrame): Aggregated generation by carrier. Index is effective_snapshots (or resampled).
            - load_dispatch_sum (pd.Series): Total load. Index is effective_snapshots (or resampled).
            - storage_dispatch_agg (pd.DataFrame): Aggregated StorageUnit charge/discharge. Index is effective_snapshots (or resampled).
            - store_dispatch_agg (pd.DataFrame): Aggregated Store charge/discharge. Index is effective_snapshots (or resampled).
    """
    effective_snapshots = get_effective_snapshots(_n, _snapshots_slice)
    if effective_snapshots.empty:
        logger.warning("get_dispatch_data: Effective snapshots are empty. Returning empty data structures.")
        # Return empty structures with the correct (empty) index type if possible
        empty_idx = pd.Index([]) if not isinstance(effective_snapshots, (pd.DatetimeIndex, pd.MultiIndex)) else effective_snapshots
        return pd.DataFrame(index=empty_idx), pd.Series(dtype=float, index=empty_idx), pd.DataFrame(index=empty_idx), pd.DataFrame(index=empty_idx)

    logger.info(f"Extracting dispatch data for {len(effective_snapshots)} snapshots. Target resolution: {resolution}")
    
    # Initialize DataFrames/Series with the final index to ensure consistency
    gen_dispatch_agg = pd.DataFrame(index=effective_snapshots)
    load_dispatch_sum = pd.Series(0.0, index=effective_snapshots) 
    storage_dispatch_agg = pd.DataFrame(index=effective_snapshots) 
    store_dispatch_agg = pd.DataFrame(index=effective_snapshots)   

    carriers_df_orig = _n.carriers if hasattr(_n, 'carriers') else None # Can be None

    component_details = {
        'generators': {'p_attr': 'p', 'target_df_name': 'gen_dispatch_agg', 'is_storage': False},
        'loads': {'p_attr': 'p_set', 'fallback_p_attr': 'p', 'target_df_name': 'load_dispatch_sum', 'is_storage': False},
        'storage_units': {'p_attr': 'p', 'target_df_name': 'storage_dispatch_agg', 'is_storage': True},
        'stores': {'p_attr': 'p', 'target_df_name': 'store_dispatch_agg', 'is_storage': True}
    }
    
    # Use a dictionary to manage the target dataframes for updating inside loop
    target_dfs = {
        'gen_dispatch_agg': gen_dispatch_agg,
        'load_dispatch_sum': load_dispatch_sum,
        'storage_dispatch_agg': storage_dispatch_agg,
        'store_dispatch_agg': store_dispatch_agg
    }

    for comp_type, details in component_details.items():
        if comp_type in _n.components and hasattr(_n, f"{comp_type}_t"):
            comp_t_panel = getattr(_n, f"{comp_type}_t", {})
            comp_t_data = comp_t_panel.get(details['p_attr'])
            
            if comp_type == 'loads' and (comp_t_data is None or comp_t_data.empty): # Fallback for loads
                 comp_t_data = comp_t_panel.get(details.get('fallback_p_attr')) 
            
            if comp_t_data is not None and not comp_t_data.empty:
                df_static = getattr(_n, comp_type)
                if df_static.empty and comp_type != 'loads': 
                    logger.debug(f"No static data for '{comp_type}', skipping dispatch sum for it.")
                    continue

                relevant_cols = df_static.index.intersection(comp_t_data.columns) if not df_static.empty else comp_t_data.columns
                if relevant_cols.empty and comp_type != 'loads': 
                    logger.debug(f"No relevant component columns for '{comp_type}' in time-series data.")
                    continue
                
                aligned_data = comp_t_data.reindex(index=effective_snapshots, columns=relevant_cols).fillna(0)

                current_target_df_name = details['target_df_name']

                if comp_type == 'generators':
                    # Ensure relevant_cols are valid for df_static if it's not empty
                    static_df_for_map = df_static.loc[relevant_cols] if not df_static.empty else pd.DataFrame(index=relevant_cols) # Dummy for carrier map if static is empty
                    carrier_map = get_carrier_map(static_df_for_map, carriers_df_orig)
                    if carrier_map is not None and not carrier_map.empty:
                        # Update the DataFrame in the target_dfs dictionary
                        target_dfs[current_target_df_name] = aligned_data.groupby(carrier_map, axis=1).sum()
                elif comp_type == 'loads':
                    target_dfs[current_target_df_name] = aligned_data.sum(axis=1) # Update Series in dict
                elif details['is_storage']:
                    default_name_base = comp_type.replace('_units', '').replace('s', '').title()
                    static_df_for_map = df_static.loc[relevant_cols] if not df_static.empty else pd.DataFrame(index=relevant_cols)
                    carrier_map = get_carrier_map(static_df_for_map, carriers_df_orig, 
                                                  default_carrier_name=f"{default_name_base} (Default)") # e.g. StorageUnit (Default)
                    if carrier_map is not None and not carrier_map.empty:
                        grouped_p_storage = aligned_data.groupby(carrier_map, axis=1).sum()
                        # Access the DataFrame from the dictionary to update it
                        temp_storage_df = target_dfs[current_target_df_name]
                        for mapped_carrier_name in grouped_p_storage.columns:
                            temp_storage_df[f"{str(mapped_carrier_name)} Discharge"] = grouped_p_storage[mapped_carrier_name].clip(lower=0)
                            temp_storage_df[f"{str(mapped_carrier_name)} Charge"] = grouped_p_storage[mapped_carrier_name].clip(upper=0) 
                        target_dfs[current_target_df_name] = temp_storage_df # Assign back if it was copied
    
    # Retrieve updated DataFrames/Series from the dictionary
    gen_dispatch_agg = target_dfs['gen_dispatch_agg']
    load_dispatch_sum = target_dfs['load_dispatch_sum']
    storage_dispatch_agg = target_dfs['storage_dispatch_agg']
    store_dispatch_agg = target_dfs['store_dispatch_agg']
    
    # Final reindex, fill NaNs, and remove columns that are all zero (or very close to it)
    gen_dispatch_agg = gen_dispatch_agg.reindex(effective_snapshots).fillna(0)
    gen_dispatch_agg = gen_dispatch_agg.loc[:, (gen_dispatch_agg.abs().sum(axis=0) > 1e-6)]
    
    load_dispatch_sum = load_dispatch_sum.reindex(effective_snapshots).fillna(0) 
    
    storage_dispatch_agg = storage_dispatch_agg.reindex(effective_snapshots).fillna(0)
    storage_dispatch_agg = storage_dispatch_agg.loc[:, (storage_dispatch_agg.abs().sum(axis=0) > 1e-6)]
    
    store_dispatch_agg = store_dispatch_agg.reindex(effective_snapshots).fillna(0)
    store_dispatch_agg = store_dispatch_agg.loc[:, (store_dispatch_agg.abs().sum(axis=0) > 1e-6)]
    
    # --- Resampling Stage ---
    if resolution != "1H": # Assuming "1H" is the base resolution from PyPSA processing
        time_index_for_resampling = get_time_index(effective_snapshots) 
        if time_index_for_resampling is not None and not time_index_for_resampling.empty:
            logger.info(f"Resampling dispatch data to {resolution}.")
            def _resample_utility(data_item, base_time_idx, res_freq):
                if data_item.empty: return data_item
                temp_data = data_item.copy()
                # Ensure index is DatetimeIndex and matches length for safe assignment
                current_time_idx = get_time_index(temp_data.index)
                if current_time_idx is None or len(current_time_idx) != len(temp_data):
                    # Fallback to base_time_idx if current is bad or mismatched
                    if len(base_time_idx) == len(temp_data):
                        temp_data.index = base_time_idx
                    else:
                        logger.warning(f"Cannot reliably resample {type(data_item).__name__} due to index mismatch. Base len: {len(base_time_idx)}, Data len: {len(temp_data)}")
                        return data_item # Return original if cannot align
                elif not isinstance(temp_data.index, pd.DatetimeIndex) : # If get_time_index worked but wasn't assigned
                     temp_data.index = current_time_idx

                if temp_data.index.empty or not isinstance(temp_data.index, pd.DatetimeIndex):
                     logger.warning(f"Skipping resampling for {type(data_item).__name__}, invalid time index post-alignment.")
                     return data_item
                return temp_data.resample(res_freq).mean() # Use mean for power averaging

            gen_dispatch_agg = _resample_utility(gen_dispatch_agg, time_index_for_resampling, resolution)
            load_dispatch_sum = _resample_utility(load_dispatch_sum, time_index_for_resampling, resolution)
            storage_dispatch_agg = _resample_utility(storage_dispatch_agg, time_index_for_resampling, resolution)
            store_dispatch_agg = _resample_utility(store_dispatch_agg, time_index_for_resampling, resolution)
        else:
            logger.warning(f"Cannot resample dispatch data to {resolution}: No valid DatetimeIndex derived from effective_snapshots.")

    return gen_dispatch_agg, load_dispatch_sum, storage_dispatch_agg, store_dispatch_agg

def dispatch_data_payload_former(n, snapshots_slice=None, resolution="1H", **kwargs) -> Dict[str, Any]:
    """
    Prepares the dispatch data payload for an API response.
    It calls `get_dispatch_data` and formats the output into a dictionary suitable for JSON.

    Args:
        n (pypsa.Network): The PyPSA network object.
        snapshots_slice: Optional pre-filtered snapshots.
        resolution (str): Target time resolution.

    Returns:
        Dict[str, Any]: A dictionary containing formatted dispatch data.
    """
    gen_dispatch, load_dispatch, storage_dispatch, store_dispatch = get_dispatch_data(
        n, _snapshots_slice=snapshots_slice, resolution=resolution
    )
    
    # Determine the primary index for timestamps from the (potentially resampled) data
    # This ensures timestamps in payload match the data's actual index.
    final_data_index = pd.Index([]) 
    if not gen_dispatch.empty: final_data_index = gen_dispatch.index
    elif isinstance(load_dispatch, pd.Series) and not load_dispatch.empty: final_data_index = load_dispatch.index 
    elif not storage_dispatch.empty: final_data_index = storage_dispatch.index
    elif not store_dispatch.empty: final_data_index = store_dispatch.index
    
    timestamps_as_strings = [str(ts) for ts in get_time_index(final_data_index)] if not final_data_index.empty else []
    
    load_data_list_of_dicts = []
    if isinstance(load_dispatch, pd.Series) and not load_dispatch.empty: 
        for timestamp_value, load_value in load_dispatch.items(): # Iterate over Series
            load_data_list_of_dicts.append({'timestamp': str(timestamp_value), 'load': load_value if pd.notna(load_value) else 0.0})
    
    # Use OrderedDict for to_dict if an older pandas/python might not preserve key order
    # For modern Python (3.7+) and Pandas, dicts are ordered by insertion.
    return {
        'generation': gen_dispatch.reset_index().to_dict('records', into=OrderedDict) if not gen_dispatch.empty else [],
        'load': load_data_list_of_dicts, 
        'storage': storage_dispatch.reset_index().to_dict('records', into=OrderedDict) if not storage_dispatch.empty else [],
        'store': store_dispatch.reset_index().to_dict('records', into=OrderedDict) if not store_dispatch.empty else [],
        'timestamps': timestamps_as_strings,
    }

def get_carrier_capacity(_n: pypsa.Network, attribute: str = "p_nom_opt", period_filter_value_str: Optional[str]=None) -> pd.DataFrame:
    """
    Calculates aggregated capacity by carrier, optionally filtered by a specific period
    for multi-period networks (based on 'build_year' and 'lifetime').

    Args:
        _n (pypsa.Network): The PyPSA network object.
        attribute (str): The capacity attribute to sum (e.g., 'p_nom_opt', 'e_nom').
        period_filter_value_str (Optional[str]): The specific period (year) to filter assets for.

    Returns:
        pd.DataFrame: DataFrame with 'Carrier', 'Capacity', and 'Unit' columns.
    """
    logger.info(f"Calculating capacity for attribute '{attribute}'" + (f" for period '{period_filter_value_str}'" if period_filter_value_str else ""))
    capacity_list = []
    carriers_df_orig = _n.carriers if hasattr(_n, 'carriers') else None # Can be None
    is_multi_period_network = isinstance(safe_get_snapshots(_n), pd.MultiIndex) and safe_get_snapshots(_n).nlevels > 0

    components_to_check = {'Generator': 'generators', 'StorageUnit': 'storage_units', 'Store': 'stores'}

    for comp_cls, comp_attr_name in components_to_check.items():
        if comp_attr_name in _n.components:
            df_comp_full = getattr(_n, comp_attr_name)
            if not df_comp_full.empty:
                attr_to_use = attribute # Start with the requested attribute
                # Adjust attribute if primary type doesn't match (e.g. energy for Store, power for Gen/SU)
                if comp_cls == 'Store': 
                    if attribute not in ['e_nom', 'e_nom_opt']: 
                        attr_to_use = 'e_nom_opt' if 'e_nom_opt' in df_comp_full.columns else 'e_nom'
                else: 
                    if attribute not in ['p_nom', 'p_nom_opt']: 
                        attr_to_use = 'p_nom_opt' if 'p_nom_opt' in df_comp_full.columns else 'p_nom'
                
                if attr_to_use not in df_comp_full.columns:
                    logger.debug(f"Chosen attribute '{attr_to_use}' not found in component '{comp_cls}'. Skipping.")
                    continue
                
                df_for_period_calculation = df_comp_full
                # If multi-period network and a specific period is requested for filtering assets
                if is_multi_period_network and period_filter_value_str is not None and \
                   'build_year' in df_comp_full.columns and 'lifetime' in df_comp_full.columns:
                    try:
                        # Attempt to convert the string period filter to the dtype of 'build_year'
                        period_val_typed_for_filter = df_comp_full['build_year'].dtype.type(period_filter_value_str)
                        active_assets_mask = (df_comp_full['build_year'] <= period_val_typed_for_filter) & \
                                             ((df_comp_full['build_year'] + df_comp_full['lifetime']) > period_val_typed_for_filter)
                        df_for_period_calculation = df_comp_full[active_assets_mask]
                    except ValueError:
                        logger.warning(f"Cannot convert period filter '{period_filter_value_str}' to type of 'build_year' for {comp_cls}. Using all assets for this component in period sum.")
                    except Exception as e_filter_assets:
                         logger.warning(f"Error filtering assets for {comp_cls} in period {period_filter_value_str}: {e_filter_assets}. Using all assets for this component.")
                
                if not df_for_period_calculation.empty:
                    carrier_map = get_carrier_map(df_for_period_calculation, carriers_df_orig, default_carrier_name=comp_cls)
                    if carrier_map is not None and not carrier_map.empty:
                        comp_capacity_sum = df_for_period_calculation.groupby(carrier_map)[attr_to_use].sum()
                        capacity_list.append(comp_capacity_sum)
    
    if capacity_list:
        combined_capacity_series = pd.concat(capacity_list).groupby(level=0).sum() 
        result_df = combined_capacity_series.reset_index(name='Capacity')
        result_df.rename(columns={'index':'Carrier'}, inplace=True) # Ensure first column is 'Carrier'
        
        # Determine unit based on the originally requested attribute for consistency with user selection
        unit_final = 'MWh' if 'e_nom' in attribute else 'MW' 
        result_df['Unit'] = unit_final
        return result_df[result_df['Capacity'].abs() > 1e-6] # Filter out negligible capacities
    else:
        return pd.DataFrame(columns=['Carrier', 'Capacity', 'Unit'])

def get_buses_capacity(_n: pypsa.Network, attribute: str = "p_nom_opt", period_filter_value_str: Optional[str]=None) -> pd.DataFrame:
    """Calculates aggregated capacity by bus/region, optionally filtered by a specific period."""
    logger.info(f"Calculating capacity by bus/region for attribute '{attribute}'" + (f" for period '{period_filter_value_str}'" if period_filter_value_str else ""))
    capacity_list_bus = []
    is_multi_period_network = isinstance(safe_get_snapshots(_n), pd.MultiIndex) and safe_get_snapshots(_n).nlevels > 0
    components_to_check = {'Generator': 'generators', 'StorageUnit': 'storage_units', 'Store': 'stores'}

    for comp_cls, comp_attr_name in components_to_check.items():
        if comp_attr_name in _n.components:
            df_comp_full = getattr(_n, comp_attr_name)
            if not df_comp_full.empty and 'bus' in df_comp_full.columns: # Crucially, group by 'bus'
                attr_to_use = attribute
                if comp_cls == 'Store':
                    if attribute not in ['e_nom', 'e_nom_opt']:
                        attr_to_use = 'e_nom_opt' if 'e_nom_opt' in df_comp_full.columns else 'e_nom'
                else: 
                    if attribute not in ['p_nom', 'p_nom_opt']:
                        attr_to_use = 'p_nom_opt' if 'p_nom_opt' in df_comp_full.columns else 'p_nom'
                
                if attr_to_use not in df_comp_full.columns:
                    logger.debug(f"Attribute '{attr_to_use}' not found in '{comp_cls}' for bus capacity. Skipping.")
                    continue
                
                df_for_period_calc = df_comp_full
                if is_multi_period_network and period_filter_value_str is not None and \
                   'build_year' in df_comp_full.columns and 'lifetime' in df_comp_full.columns:
                    try:
                        period_val_typed = df_comp_full['build_year'].dtype.type(period_filter_value_str)
                        active_mask = (df_comp_full['build_year'] <= period_val_typed) & \
                                      ((df_comp_full['build_year'] + df_comp_full['lifetime']) > period_val_typed)
                        df_for_period_calc = df_comp_full[active_mask]
                    except Exception: pass # If filtering fails, use all assets for this component type for this period
                
                if not df_for_period_calc.empty:
                    # Group by 'bus' and sum the chosen capacity attribute
                    comp_capacity_by_bus_sum = df_for_period_calc.groupby(df_for_period_calc['bus'])[attr_to_use].sum()
                    capacity_list_bus.append(comp_capacity_by_bus_sum)
    
    if capacity_list_bus:
        combined_capacity_by_bus_series = pd.concat(capacity_list_bus).groupby(level=0).sum()
        result_df = combined_capacity_by_bus_series.reset_index(name='Capacity')
        result_df.rename(columns={'index':'Region'}, inplace=True) # 'Region' here refers to bus name
        unit_final = 'MWh' if 'e_nom' in attribute else 'MW'
        result_df['Unit'] = unit_final
        return result_df[result_df['Capacity'].abs() > 1e-6]
    else:
        return pd.DataFrame(columns=['Region', 'Capacity', 'Unit'])

def carrier_capacity_payload_former(n, snapshots_slice=None, attribute="p_nom_opt", **kwargs) -> Dict[str, List[Dict]]:
    """
    Prepares capacity data payload (by carrier and by region) for the API.
    'period' from URL query args is used for asset filtering in multi-period networks.
    `snapshots_slice` is not directly used by these capacity functions as they rely on static data.
    """
    period_for_asset_filtering = kwargs.get('period') # This is the string period name from URL
    
    df_capacity_by_carrier = get_carrier_capacity(n, attribute=attribute, period_filter_value_str=period_for_asset_filtering)
    df_capacity_by_region = get_buses_capacity(n, attribute=attribute, period_filter_value_str=period_for_asset_filtering)
    
    return {
        'by_carrier': df_capacity_by_carrier.to_dict('records', into=OrderedDict) if not df_capacity_by_carrier.empty else [],
        'by_region': df_capacity_by_region.to_dict('records', into=OrderedDict) if not df_capacity_by_region.empty else [],
    }
import pypsa
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import logging
from typing import Union, Optional, Tuple, Dict, List, Any
from plotly.subplots import make_subplots
from collections import OrderedDict # For to_dict('records', into=OrderedDict)

# Logging configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Default color palette
DEFAULT_COLORS = {
    'Coal': '#000000', 'coal': '#000000',
    'Lignite': '#4B4B4B', 'lignite': '#4B4B4B',
    'Nuclear': '#800080', 'nuclear': '#800080',
    'Hydro': '#0073CF', 'hydro': '#0073CF', 'hydro ror': '#3399FF',
    'Hydro RoR': '#3399FF', 'ror': '#3399FF', 'Hydro Storage': '#3399FF', # Ensure "Hydro RoR" takes precedence
    'Solar': '#FFD700', 'solar': '#FFD700', 'pv': '#FFD700', 'Solar PV': '#FFD700',
    'Wind': '#ADD8E6', 'wind': '#ADD8E6', 'onwind': '#ADD8E6', 'offwind': '#ADD8E6', 'Onshore Wind': '#ADD8E6', 'Offshore Wind': '#6495ED',
    'LFO': '#FF4500', 'lfo': '#FF4500', 'Oil': '#FF4500', 'oil': '#FF4500', 'Diesel': '#FF4500',
    'Co-Gen': '#228B22', 'co-gen': '#228B22', 'biomass': '#228B22', 'Biomass': '#228B22',
    'PSP': '#3399FF', 'psp': '#3399FF', 'Pumped Hydro': '#3399FF',
    'Battery Storage': '#005B5B', 'battery': '#005B5B', 'Battery': '#005B5B',
    'Planned Battery Storage': '#66B2B2', 'planned battery': '#66B2B2',
    'Planned PSP': '#B0C4DE', 'planned psp': '#B0C4DE',
    'Storage': '#B0C4DE', # Generic storage
    'H2 Storage': '#AFEEEE', 'hydrogen': '#AFEEEE', 'h2': '#AFEEEE', 'H2': '#AFEEEE', 'Hydrogen Storage': '#AFEEEE',
    'Load': '#000000',
    'Transmission': '#808080', 'Line': '#808080', 'Link': '#A9A9A9',
    'Losses': '#DC143C',
    'Other': '#D3D3D3',
    'Curtailment': '#FF00FF',
    'Excess': '#FF00FF', # Often interchangeable with curtailment
    'Storage Charge': '#FFA500', # Generic charge
    'Storage Discharge': '#50C878', # Generic discharge (a green)
    'Store Charge': '#AFEEEE', # Default for 'Store' type if no specific carrier
    'Store Discharge': '#87CEEB', # Default for 'Store' type
}

PLOTLY_COLOR_CYCLE = px.colors.qualitative.Plotly

# --- Utility Functions for Snapshot and Index Handling ---
def safe_get_snapshots(n: pypsa.Network) -> Union[pd.DatetimeIndex, pd.MultiIndex, pd.Index]:
    """Safely get network snapshots, returning an empty pd.Index if not available."""
    return n.snapshots if hasattr(n, 'snapshots') and n.snapshots is not None else pd.Index([])

def get_time_index(index: Union[pd.DatetimeIndex, pd.MultiIndex, pd.Index, None]) -> Optional[pd.DatetimeIndex]:
    """
    Extracts or converts the time component of a pandas Index to a DatetimeIndex.
    Returns None if conversion is not possible or index is empty.
    """
    if index is None or index.empty:
        return None
    if isinstance(index, pd.DatetimeIndex):
        return index
    
    target_index_level = index
    if isinstance(index, pd.MultiIndex):
        if index.nlevels > 0:
            target_index_level = index.get_level_values(-1) # Assume time is the last level
        else: # Should not happen for valid MultiIndex
            return None
            
    if pd.api.types.is_datetime64_any_dtype(target_index_level):
        return pd.DatetimeIndex(target_index_level)
    else:
        try:
            # Attempt conversion, handling potential errors for non-convertible types
            converted_index = pd.to_datetime(target_index_level, errors='coerce')
            if converted_index.hasnans and not pd.Series(target_index_level).hasnans: # Check if coercion introduced NaNs
                 logging.warning(f"Conversion to DatetimeIndex introduced NaNs for index type {type(target_index_level)}. Original may not be time-like.")
                 return None
            return converted_index
        except (TypeError, ValueError) as e:
            logging.warning(f"Could not convert index of type {type(target_index_level)} to DatetimeIndex: {e}")
            return None

def get_period_index(index: Union[pd.DatetimeIndex, pd.MultiIndex, pd.Index, None]) -> Optional[Union[pd.Index, pd.Series]]:
    """
    Extracts the period component from a pandas Index.
    For DatetimeIndex, assumes annual periods. For MultiIndex, takes the first level.
    Returns None if not applicable or index is empty.
    """
    if index is None or index.empty:
        return None
    if isinstance(index, pd.MultiIndex):
        if index.nlevels > 0:
            return index.get_level_values(0) # Assume period is the first level
        else:
            return None # Should not happen
    elif isinstance(index, pd.DatetimeIndex):
        return pd.Series(index.year, index=index) # Simple annual period for DatetimeIndex
    
    logging.warning(f"Cannot determine period index from type {type(index)}. Returning None.")
    return None

def get_snapshot_weights(n: pypsa.Network, snapshots_idx: Union[pd.DatetimeIndex, pd.MultiIndex, pd.Index]) -> pd.Series:
    """
    Get snapshot weights, aligning with the provided snapshots_idx.
    Defaults to 1.0 if weights are not available or don't align.
    """
    if snapshots_idx is None or snapshots_idx.empty:
        return pd.Series(dtype=float) # Return empty Series if no snapshots
        
    if hasattr(n, 'snapshot_weightings') and not n.snapshot_weightings.empty and 'objective' in n.snapshot_weightings.columns:
        weights = n.snapshot_weightings.objective
        # Align weights with the potentially filtered/sliced snapshots_idx
        common_index = snapshots_idx.intersection(weights.index)
        if not common_index.empty:
            return weights.loc[common_index].reindex(snapshots_idx).fillna(1.0)
        else:
            logging.warning("No common index between provided snapshots and network's snapshot_weightings. Assuming weight 1.0.")
    else:
        logging.warning("Snapshot weights ('objective' column) not found or empty in network.snapshot_weightings. Assuming weight 1.0 for all snapshots.")
    return pd.Series(1.0, index=snapshots_idx)

def get_effective_snapshots(n: pypsa.Network, _snapshots_slice: Optional[Union[pd.DatetimeIndex, pd.MultiIndex, pd.Index]] = None) -> Union[pd.DatetimeIndex, pd.MultiIndex, pd.Index]:
    """
    Determines the effective set of snapshots to use for calculations.
    If _snapshots_slice is provided and valid, it's used. Otherwise, falls back to network's snapshots.
    Returns an empty pd.Index if no valid snapshots can be determined.
    """
    if _snapshots_slice is not None:
        if not _snapshots_slice.empty:
            return _snapshots_slice
        else:
            # If an empty slice was explicitly passed, it means no data for that selection
            logging.debug("get_effective_snapshots: Received an explicitly empty _snapshots_slice.")
            return pd.Index([]) 
    # Fallback to network's full snapshots if no slice provided
    return safe_get_snapshots(n)

def get_carrier_map(comp_df: pd.DataFrame, carriers_df: Optional[pd.DataFrame], default_carrier_name: Optional[str] = None) -> Optional[pd.Series]:
    """
    Helper function to get a mapping from component names to 'nice' carrier names.
    Uses 'nice_name' from carriers_df if available, otherwise defaults to original carrier or a provided default.
    """
    if 'carrier' not in comp_df.columns and default_carrier_name is None:
        logging.debug(f"Component DataFrame does not have 'carrier' column and no default_carrier_name provided.")
        return None # No basis for carrier mapping
    
    # Start with the 'carrier' column or a default series if 'carrier' is missing but default_carrier_name is given
    carrier_map_series = comp_df.get('carrier', pd.Series(default_carrier_name, index=comp_df.index))
    carrier_map_series = carrier_map_series.copy() # Work on a copy to avoid SettingWithCopyWarning

    # Ensure carriers_df is usable, even if None or empty
    if not isinstance(carriers_df, pd.DataFrame) or carriers_df.empty:
        # Create a minimal carriers_df if none provided, using unique values from carrier_map_series
        unique_carriers_in_comp = carrier_map_series.dropna().unique()
        carriers_df_internal = pd.DataFrame(index=unique_carriers_in_comp)
    else:
        carriers_df_internal = carriers_df.copy()

    if 'nice_name' not in carriers_df_internal.columns:
        carriers_df_internal['nice_name'] = carriers_df_internal.index # Default nice_name to carrier index itself

    nice_name_map_dict = carriers_df_internal['nice_name'].dropna().to_dict()
    
    # Map to nice names, then fill any NaNs (unmapped original carriers) with their original names
    # This handles cases where a carrier in comp_df might not be in carriers_df
    original_carriers = carrier_map_series.copy()
    carrier_map_series = carrier_map_series.map(nice_name_map_dict)
    carrier_map_series.fillna(original_carriers, inplace=True)

    # If a default_carrier_name was provided, ensure any remaining NaNs (e.g. if original carrier was NaN) are filled
    if default_carrier_name:
         carrier_map_series.fillna(default_carrier_name, inplace=True)
            
    return carrier_map_series

# --- Core Data Extraction Functions ---
def get_dispatch_data(_n: pypsa.Network, _snapshots_slice: Optional[Union[pd.DatetimeIndex, pd.MultiIndex, pd.Index]] = None,
                     resolution: str = "1H") -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.DataFrame]:
    """
    Extracts dispatch data for generators, loads, storage units, and stores,
    aligned to effective_snapshots and optionally resampled.
    """
    effective_snapshots = get_effective_snapshots(_n, _snapshots_slice)
    if effective_snapshots.empty:
        logging.warning("get_dispatch_data: Effective snapshots are empty. Returning empty data structures.")
        return pd.DataFrame(), pd.Series(dtype=float), pd.DataFrame(), pd.DataFrame()

    logging.info(f"get_dispatch_data: Extracting dispatch data for {len(effective_snapshots)} effective_snapshots, resolution: {resolution}.")
    
    # Initialize return dataframes/series with the effective_snapshots index to ensure alignment
    gen_dispatch_agg = pd.DataFrame(index=effective_snapshots)
    load_dispatch_sum = pd.Series(0.0, index=effective_snapshots) 
    storage_units_dispatch_agg = pd.DataFrame(index=effective_snapshots)
    stores_dispatch_agg = pd.DataFrame(index=effective_snapshots)

    carriers_df = _n.carriers if hasattr(_n, 'carriers') and isinstance(_n.carriers, pd.DataFrame) else pd.DataFrame()

    # Component processing configuration
    component_configs = {
        'generators': {'df_out': gen_dispatch_agg, 'p_attr': 'p', 'default_carrier': 'Generator'},
        'loads': {'series_out': load_dispatch_sum, 'p_attr': 'p_set', 'fallback_p_attr': 'p'}, # Loads sum up
        'storage_units': {'df_out': storage_units_dispatch_agg, 'p_attr': 'p', 'default_carrier': 'StorageUnit'},
        'stores': {'df_out': stores_dispatch_agg, 'p_attr': 'p', 'default_carrier': 'Store'},
    }

    for comp_name_plural, config in component_configs.items():
        if comp_name_plural in _n.components and hasattr(_n, f"{comp_name_plural}_t"):
            df_static = getattr(_n, comp_name_plural, pd.DataFrame())
            if df_static.empty: continue

            p_attr_to_use = config['p_attr']
            comp_t_dispatch_data = getattr(_n, f"{comp_name_plural}_t", {}).get(p_attr_to_use)
            if comp_t_dispatch_data is None and 'fallback_p_attr' in config: # Try fallback for loads
                p_attr_to_use = config['fallback_p_attr']
                comp_t_dispatch_data = getattr(_n, f"{comp_name_plural}_t", {}).get(p_attr_to_use)

            if comp_t_dispatch_data is not None and not comp_t_dispatch_data.empty:
                # Align component time-series data with effective_snapshots (index and columns)
                aligned_dispatch_data = comp_t_dispatch_data.reindex(index=effective_snapshots, columns=df_static.index).fillna(0)

                if 'series_out' in config: # For loads
                    config['series_out'].update(aligned_dispatch_data.sum(axis=1))
                elif 'df_out' in config: # For generators, storage_units, stores
                    carrier_map = get_carrier_map(df_static, carriers_df, default_carrier_name=config.get('default_carrier'))
                    if carrier_map is not None:
                        # Ensure carrier_map only contains columns present in aligned_dispatch_data
                        valid_cols_for_grouping = aligned_dispatch_data.columns.intersection(carrier_map.index)
                        if not valid_cols_for_grouping.empty:
                            grouped_dispatch = aligned_dispatch_data[valid_cols_for_grouping].groupby(
                                carrier_map.loc[valid_cols_for_grouping], axis=1
                            ).sum()
                            
                            # For storage-like components, split into Charge/Discharge
                            if comp_name_plural in ['storage_units', 'stores']:
                                for carrier in grouped_dispatch.columns:
                                    config['df_out'][f"{carrier} Discharge"] = grouped_dispatch[carrier].clip(lower=0)
                                    config['df_out'][f"{carrier} Charge"] = grouped_dispatch[carrier].clip(upper=0) # Negative is charge
                            else: # For generators
                                config['df_out'].update(grouped_dispatch)
    
    # Filter out all-zero columns that might have been created
    gen_dispatch_agg = gen_dispatch_agg.loc[:, (gen_dispatch_agg.abs() > 1e-6).any(axis=0)]
    storage_units_dispatch_agg = storage_units_dispatch_agg.loc[:, (storage_units_dispatch_agg.abs() > 1e-6).any(axis=0)]
    stores_dispatch_agg = stores_dispatch_agg.loc[:, (stores_dispatch_agg.abs() > 1e-6).any(axis=0)]
    
    # Resampling if required
    if resolution != "1H":
        time_index_for_resampling = get_time_index(effective_snapshots)
        if time_index_for_resampling is not None and not time_index_for_resampling.empty:
            def resample_if_exists(df_or_series, base_idx):
                if df_or_series.empty: return df_or_series
                temp = df_or_series.copy()
                if not isinstance(temp.index, pd.DatetimeIndex): # Ensure datetime index for resampling
                    temp.index = get_time_index(base_idx) # Use the original effective_snapshots time part
                return temp.resample(resolution).mean()

            gen_dispatch_agg = resample_if_exists(gen_dispatch_agg, effective_snapshots)
            load_dispatch_sum = resample_if_exists(load_dispatch_sum, effective_snapshots)
            storage_units_dispatch_agg = resample_if_exists(storage_units_dispatch_agg, effective_snapshots)
            stores_dispatch_agg = resample_if_exists(stores_dispatch_agg, effective_snapshots)
        else:
            logging.warning(f"Cannot resample data to {resolution}. Valid DatetimeIndex not available from effective_snapshots.")

    return gen_dispatch_agg, load_dispatch_sum, storage_units_dispatch_agg, stores_dispatch_agg

def get_carrier_capacity(_n: pypsa.Network, attribute: str = "p_nom_opt", period_val_for_assets: Optional[Any] = None) -> pd.DataFrame:
    """
    Calculates aggregated capacity by carrier for specified components and attribute.
    `period_val_for_assets` is used for filtering assets based on build_year/lifetime in multi-period investment models.
    It is NOT for slicing time-series snapshots.
    """
    logging.info(f"get_carrier_capacity: Calculating for attribute '{attribute}'" + 
                 (f" considering assets active in period '{period_val_for_assets}'" if period_val_for_assets is not None else ""))
    
    capacity_data_list = []
    carriers_df = _n.carriers if hasattr(_n, 'carriers') and isinstance(_n.carriers, pd.DataFrame) else pd.DataFrame()
    components_to_check = {'Generator': 'generators', 'StorageUnit': 'storage_units', 'Store': 'stores'}

    for comp_class_name, comp_attr_name_in_n in components_to_check.items():
        if comp_attr_name_in_n in _n.components:
            df_component_static = getattr(_n, comp_attr_name_in_n, pd.DataFrame())
            if df_component_static.empty: continue

            # Determine the correct capacity attribute to use (p_nom vs e_nom for Stores)
            attr_to_use = attribute
            if comp_class_name == 'Store' and attribute not in ['e_nom', 'e_nom_opt']:
                attr_to_use = 'e_nom_opt' if 'e_nom_opt' in df_component_static.columns else 'e_nom'
            elif comp_class_name != 'Store' and attribute not in ['p_nom', 'p_nom_opt']:
                attr_to_use = 'p_nom_opt' if 'p_nom_opt' in df_component_static.columns else 'p_nom'

            if attr_to_use not in df_component_static.columns:
                logging.debug(f"Attribute '{attr_to_use}' not found in component '{comp_class_name}'. Skipping.")
                continue
            
            df_active_assets = df_component_static
            # Filter assets if period_val_for_assets is provided (for multi-investment period models)
            if period_val_for_assets is not None and 'build_year' in df_component_static.columns and 'lifetime' in df_component_static.columns:
                try:
                    # Ensure period_val_for_assets is compatible type with build_year
                    typed_period = type(df_component_static['build_year'].iloc[0])(period_val_for_assets)
                    active_mask = (df_component_static['build_year'] <= typed_period) & \
                                  ((df_component_static['build_year'] + df_component_static['lifetime']) > typed_period)
                    df_active_assets = df_component_static[active_mask]
                except Exception as e_filter:
                    logging.warning(f"Could not filter active assets for {comp_class_name} in period {period_val_for_assets} due to: {e_filter}. Using all assets.")
            
            if not df_active_assets.empty:
                carrier_map = get_carrier_map(df_active_assets, carriers_df, default_carrier_name=comp_class_name)
                if carrier_map is not None:
                    # Sum capacity for the determined attribute, grouped by mapped carrier name
                    summed_capacity_by_carrier = df_active_assets.groupby(carrier_map)[attr_to_use].sum()
                    capacity_data_list.append(summed_capacity_by_carrier)
    
    if capacity_data_list:
        # Combine capacities from different component types (e.g., a carrier might be in Generator and StorageUnit)
        final_combined_capacity = pd.concat(capacity_data_list).groupby(level=0).sum()
        result_df = final_combined_capacity.reset_index()
        result_df.columns = ['Carrier', 'Capacity']
        
        unit_for_attribute = 'MWh' if 'e_nom' in attribute else 'MW' # Determine unit based on attribute name
        result_df['Unit'] = unit_for_attribute
        result_df = result_df[result_df['Capacity'].abs() > 1e-6] # Filter out negligible capacities
        return result_df
    else:
        return pd.DataFrame(columns=['Carrier', 'Capacity', 'Unit']) # Return empty DataFrame with schema

def get_buses_capacity(_n: pypsa.Network, attribute: str = "p_nom_opt", period_val_for_assets: Optional[Any] = None) -> pd.DataFrame:
    """
    Calculates aggregated capacity by bus (region) for specified components and attribute.
    `period_val_for_assets` is used for filtering assets based on build_year/lifetime.
    """
    logging.info(f"get_buses_capacity: Calculating for attribute '{attribute}' by bus" + 
                 (f" considering assets active in period '{period_val_for_assets}'" if period_val_for_assets is not None else ""))
    
    capacity_data_list = []
    components_to_check = {'Generator': 'generators', 'StorageUnit': 'storage_units', 'Store': 'stores', 'Load': 'loads'} # Include Loads

    for comp_class_name, comp_attr_name_in_n in components_to_check.items():
        if comp_attr_name_in_n in _n.components:
            df_component_static = getattr(_n, comp_attr_name_in_n, pd.DataFrame())
            if df_component_static.empty or 'bus' not in df_component_static.columns: continue

            attr_to_use = attribute
            if comp_class_name == 'Store' and attribute not in ['e_nom', 'e_nom_opt']:
                attr_to_use = 'e_nom_opt' if 'e_nom_opt' in df_component_static.columns else 'e_nom'
            elif comp_class_name == 'Load': # Loads typically use p_set for demand capacity
                 attr_to_use = 'p_set' if 'p_set' in df_component_static.columns else attribute # Fallback if p_set not there
            elif comp_class_name not in ['Store', 'Load'] and attribute not in ['p_nom', 'p_nom_opt']:
                attr_to_use = 'p_nom_opt' if 'p_nom_opt' in df_component_static.columns else 'p_nom'

            if attr_to_use not in df_component_static.columns:
                logging.debug(f"Attribute '{attr_to_use}' not found in component '{comp_class_name}' for bus capacity. Skipping.")
                continue
            
            df_active_assets = df_component_static
            if period_val_for_assets is not None and 'build_year' in df_component_static.columns and 'lifetime' in df_component_static.columns:
                try:
                    typed_period = type(df_component_static['build_year'].iloc[0])(period_val_for_assets)
                    active_mask = (df_component_static['build_year'] <= typed_period) & \
                                  ((df_component_static['build_year'] + df_component_static['lifetime']) > typed_period)
                    df_active_assets = df_component_static[active_mask]
                except Exception as e_filter:
                    logging.warning(f"Could not filter active assets for {comp_class_name} by bus in period {period_val_for_assets}: {e_filter}. Using all.")
            
            if not df_active_assets.empty:
                # Group by 'bus' column to sum capacity per bus
                summed_capacity_by_bus = df_active_assets.groupby('bus')[attr_to_use].sum()
                capacity_data_list.append(summed_capacity_by_bus)
    
    if capacity_data_list:
        final_combined_capacity_by_bus = pd.concat(capacity_data_list).groupby(level=0).sum()
        result_df = final_combined_capacity_by_bus.reset_index()
        result_df.columns = ['Region', 'Capacity'] # 'Region' is used in JS for bus capacity plot
        
        unit_for_attribute = 'MWh' if 'e_nom' in attribute or (comp_class_name == 'Load' and 'e_' in attr_to_use) else 'MW'
        result_df['Unit'] = unit_for_attribute
        result_df = result_df[result_df['Capacity'].abs() > 1e-6]
        return result_df
    else:
        return pd.DataFrame(columns=['Region', 'Capacity', 'Unit'])

def calculate_cuf(n, snapshots_slice=None, **kwargs):
    effective_snapshots = get_effective_snapshots(n, snapshots_slice)
    if effective_snapshots.empty:
        return pd.DataFrame(columns=['Carrier', 'CUF'])

    logging.info(f"Calculating CUFs for {len(effective_snapshots)} snapshots...")
    if 'generators' not in n.components or n.generators.empty or \
       not hasattr(n, 'generators_t') or 'p' not in n.generators_t or \
       not any(c in n.generators.columns for c in ['p_nom_opt', 'p_nom']) or \
       'carrier' not in n.generators.columns:
        logging.warning("Missing data for CUF calculation.")
        return pd.DataFrame(columns=['Carrier', 'CUF'])

    try:
        # Align generators_t.p with the effective_snapshots
        gen_p_aligned = n.generators_t['p'].reindex(index=effective_snapshots, columns=n.generators.index).fillna(0)
        
        p_nom_attr = 'p_nom_opt' if 'p_nom_opt' in n.generators.columns else 'p_nom'
        gen_p_nom = n.generators[p_nom_attr] # This is a Series indexed by generator name

        weights = get_snapshot_weights(n, effective_snapshots) # Weights Series aligned with effective_snapshots
        
        # Energy produced by each generator over the effective_snapshots period
        energy_produced_per_gen = gen_p_aligned.multiply(weights, axis=0).sum(axis=0) # Sum over time (axis=0)
        
        total_hours_equivalent = weights.sum() # Sum of weights gives total equivalent hours
        if total_hours_equivalent == 0: 
            logging.warning("Total snapshot weight is zero, cannot calculate CUF.")
            return pd.DataFrame(columns=['Carrier', 'CUF'])

        # Potential energy by each generator if it ran at p_nom for total_hours_equivalent
        potential_energy_per_gen = gen_p_nom * total_hours_equivalent
        
        # CUF for each generator
        cuf_per_generator = (energy_produced_per_gen / potential_energy_per_gen.replace(0, np.nan)).fillna(0)
        cuf_per_generator = cuf_per_generator[cuf_per_generator.abs() > 1e-6] # Filter out negligible/zero CUFs

        carrier_map = get_carrier_map(n.generators, n.carriers if hasattr(n, 'carriers') else pd.DataFrame())
        if carrier_map is None or cuf_per_generator.empty: 
            return pd.DataFrame(columns=['Carrier', 'CUF'])
        
        # Average CUF by carrier, considering only generators that had some CUF
        # Ensure we only group by carriers of generators that are in cuf_per_generator
        valid_carrier_map_for_cuf = carrier_map.loc[carrier_map.index.intersection(cuf_per_generator.index)]
        if valid_carrier_map_for_cuf.empty:
             return pd.DataFrame(columns=['Carrier', 'CUF'])
        cuf_by_carrier = cuf_per_generator.groupby(valid_carrier_map_for_cuf).mean()
        
        cuf_df = cuf_by_carrier.reset_index()
        cuf_df.columns = ['Carrier', 'CUF']
        return cuf_df[cuf_df['CUF'].notna()] # Ensure no NaN CUFs are returned
    except Exception as e:
        logging.error(f"Error calculating CUFs: {e}", exc_info=True)
        return pd.DataFrame(columns=['Carrier', 'CUF'])

def calculate_curtailment(n, snapshots_slice=None, **kwargs):
    effective_snapshots = get_effective_snapshots(n, snapshots_slice)
    if effective_snapshots.empty:
        return pd.DataFrame(columns=['Carrier', 'Curtailment (MWh)', 'Potential (MWh)', 'Curtailment (%)'])
        
    logging.info(f"Calculating curtailment for {len(effective_snapshots)} snapshots...")
    req_cols_t = ['p', 'p_max_pu'] # Time-dependent columns needed
    if 'generators' not in n.components or n.generators.empty or \
       not hasattr(n, 'generators_t') or not all(c in n.generators_t for c in req_cols_t) or \
       'carrier' not in n.generators.columns or \
       not any(c in n.generators.columns for c in ['p_nom_opt', 'p_nom']): # Static capacity needed
        logging.warning("Missing essential data for curtailment calculation (generators, p, p_max_pu, p_nom/p_nom_opt, carrier).")
        return pd.DataFrame(columns=['Carrier', 'Curtailment (MWh)', 'Potential (MWh)', 'Curtailment (%)'])

    try:
        # Identify renewable generators (this might need to be more robust based on your carrier names)
        renewable_keywords = ['solar', 'wind', 'ror'] # Hydro RoR is often considered curtailable
        # Ensure carrier names are strings for matching
        n.generators['carrier_str'] = n.generators['carrier'].astype(str)
        renewable_carriers_names = [c for c in n.generators['carrier_str'].dropna().unique() if any(k in c.lower() for k in renewable_keywords)]
        
        renewable_gens_df = n.generators[n.generators['carrier_str'].isin(renewable_carriers_names)]
        if renewable_gens_df.empty:
            logging.info("No renewable generators found based on keywords. No curtailment to calculate.")
            return pd.DataFrame(columns=['Carrier', 'Curtailment (MWh)', 'Potential (MWh)', 'Curtailment (%)'])

        p_nom_attr = 'p_nom_opt' if 'p_nom_opt' in renewable_gens_df.columns else 'p_nom'
        p_nom_renewable = renewable_gens_df[p_nom_attr] # Series of nominal capacities for renewable gens

        # Align time-series data with effective_snapshots and relevant generator columns
        p_actual_aligned = n.generators_t['p'].reindex(index=effective_snapshots, columns=renewable_gens_df.index).fillna(0)
        p_max_pu_aligned = n.generators_t['p_max_pu'].reindex(index=effective_snapshots, columns=renewable_gens_df.index).fillna(0)
        
        weights = get_snapshot_weights(n, effective_snapshots) # Weights aligned with effective_snapshots
        
        # Calculate potential power (MW) for each renewable generator at each snapshot
        p_potential_mw = p_max_pu_aligned.multiply(p_nom_renewable.reindex(p_max_pu_aligned.columns), axis=1)
        
        # Calculate curtailment power (MW) = Potential - Actual, cannot be negative
        curtailment_power_mw = (p_potential_mw - p_actual_aligned).clip(lower=0)

        # Calculate energy (MWh) by multiplying power (MW) with snapshot weights (hours)
        curtailment_energy_mwh_per_gen = curtailment_power_mw.multiply(weights, axis=0).sum(axis=0) # Sum over time
        potential_energy_mwh_per_gen = p_potential_mw.multiply(weights, axis=0).sum(axis=0) # Sum over time

        carrier_map = get_carrier_map(renewable_gens_df, n.carriers if hasattr(n, 'carriers') else pd.DataFrame())
        if carrier_map is None: 
            return pd.DataFrame(columns=['Carrier', 'Curtailment (MWh)', 'Potential (MWh)', 'Curtailment (%)'])

        # Group by mapped carrier name
        curtailment_by_carrier_mwh = curtailment_energy_mwh_per_gen.groupby(carrier_map.loc[curtailment_energy_mwh_per_gen.index]).sum()
        potential_by_carrier_mwh = potential_energy_mwh_per_gen.groupby(carrier_map.loc[potential_energy_mwh_per_gen.index]).sum()
        
        curtailment_results_df = pd.DataFrame({
            'Carrier': curtailment_by_carrier_mwh.index,
            'Curtailment (MWh)': curtailment_by_carrier_mwh.values,
            'Potential (MWh)': potential_by_carrier_mwh.reindex(curtailment_by_carrier_mwh.index).fillna(0).values # Align and fill for carriers with curtailment but maybe no potential if filtered
        })
        # Calculate percentage, avoid division by zero
        curtailment_results_df['Curtailment (%)'] = (curtailment_results_df['Curtailment (MWh)'] / curtailment_results_df['Potential (MWh)'].replace(0, np.nan) * 100).fillna(0)
        
        return curtailment_results_df[curtailment_results_df['Potential (MWh)'].abs() > 1e-3] # Filter if potential is negligible
    except Exception as e:
        logging.error(f"Error calculating curtailment: {e}", exc_info=True)
        return pd.DataFrame(columns=['Carrier', 'Curtailment (MWh)', 'Potential (MWh)', 'Curtailment (%)'])

def get_storage_soc(n: pypsa.Network, snapshots_slice=None) -> pd.DataFrame:
    """
    Extracts State of Charge (SoC) for storage_units and e (energy) for stores,
    aligned to effective_snapshots.
    """
    effective_snapshots = get_effective_snapshots(n, snapshots_slice)
    if effective_snapshots.empty:
        return pd.DataFrame()

    logging.info(f"get_storage_soc: Extracting SoC/energy for {len(effective_snapshots)} snapshots...")
    soc_data_frames_list = [] # Store DataFrames for each component type before concat
    carriers_df = n.carriers if hasattr(n, 'carriers') and isinstance(n.carriers, pd.DataFrame) else pd.DataFrame()

    # Configuration for storage-like components
    storage_components_config = {
        'storage_units': {'soc_attr': 'state_of_charge', 'default_carrier_suffix': 'StorageUnit'},
        'stores': {'soc_attr': 'e', 'default_carrier_suffix': 'Store'},
    }

    for comp_name_plural, config in storage_components_config.items():
        if comp_name_plural in n.components and hasattr(n, f"{comp_name_plural}_t"):
            df_static = getattr(n, comp_name_plural, pd.DataFrame())
            if df_static.empty: continue

            soc_attr_name = config['soc_attr']
            comp_t_soc_data = getattr(n, f"{comp_name_plural}_t", {}).get(soc_attr_name)

            if comp_t_soc_data is not None and not comp_t_soc_data.empty:
                # Align SoC data with effective_snapshots (index and columns)
                aligned_soc_data = comp_t_soc_data.reindex(index=effective_snapshots, columns=df_static.index).fillna(0)
                
                # Get carrier mapping, using a suffix to distinguish if a carrier is used for different storage types
                carrier_map = get_carrier_map(df_static, carriers_df, default_carrier_name=f"Default {config['default_carrier_suffix']}")
                if carrier_map is not None:
                    # Suffix the mapped carrier names to make them unique per component type
                    # e.g., "Battery (StorageUnit)" vs "Hydrogen (Store)"
                    suffixed_carrier_map = carrier_map.apply(lambda x: f"{x} ({config['default_carrier_suffix']})")
                    
                    valid_cols_for_grouping = aligned_soc_data.columns.intersection(suffixed_carrier_map.index)
                    if not valid_cols_for_grouping.empty:
                        grouped_soc_data = aligned_soc_data[valid_cols_for_grouping].groupby(
                            suffixed_carrier_map.loc[valid_cols_for_grouping], axis=1
                        ).sum()
                        soc_data_frames_list.append(grouped_soc_data)
    
    if not soc_data_frames_list: 
        return pd.DataFrame(index=effective_snapshots) # Return empty DF with correct index
        
    # Concatenate all SoC/energy dataframes, then reindex again to ensure full effective_snapshots index
    combined_soc_df = pd.concat(soc_data_frames_list, axis=1).reindex(effective_snapshots).fillna(0)
    return combined_soc_df.loc[:, (combined_soc_df.abs() > 1e-6).any(axis=0)] # Filter out all-zero columns

def calculate_co2_emissions(n, snapshots_slice=None, **kwargs):
    effective_snapshots = get_effective_snapshots(n, snapshots_slice)
    empty_total_df = pd.DataFrame(columns=['Period', 'Total CO2 Emissions (Tonnes)'])
    empty_carrier_df = pd.DataFrame(columns=['Period', 'Carrier', 'Emissions (Tonnes)'])
    if effective_snapshots.empty:
        return empty_total_df, empty_carrier_df

    logging.info(f"Calculating CO2 emissions for {len(effective_snapshots)} snapshots...")
    
    if 'generators' not in n.components or n.generators.empty or \
       not hasattr(n, 'generators_t') or 'p' not in n.generators_t or \
       not hasattr(n, 'carriers') or 'co2_emissions' not in n.carriers.columns:
        logging.warning("Missing data for CO2 emissions (generators, p, carriers.co2_emissions).")
        return empty_total_df, empty_carrier_df

    try:
        co2_factors_per_carrier = n.carriers['co2_emissions'].dropna() # Series: carrier -> co2_factor
        if co2_factors_per_carrier.empty: 
            logging.info("No CO2 emission factors defined in n.carriers.co2_emissions.")
            return empty_total_df, empty_carrier_df

        # Filter generators that have carriers with defined CO2 factors
        emitting_gens_df = n.generators[n.generators['carrier'].isin(co2_factors_per_carrier.index)]
        if emitting_gens_df.empty: 
            logging.info("No generators found with carriers that have CO2 emission factors.")
            return empty_total_df, empty_carrier_df

        # Align generation data (p) with effective_snapshots and relevant generators
        gen_p_aligned = n.generators_t.p.reindex(index=effective_snapshots, columns=emitting_gens_df.index).fillna(0)
        weights = get_snapshot_weights(n, effective_snapshots) # Weights aligned with effective_snapshots

        # Map CO2 factors to each generator based on its carrier
        co2_factors_for_gens = emitting_gens_df['carrier'].map(co2_factors_per_carrier) # Series: generator_name -> co2_factor
        
        # Calculate emissions (Tonnes) for each generator at each snapshot: Power (MW) * Factor (tCO2/MWh) * Weight (h)
        emissions_timeseries_per_gen = gen_p_aligned.multiply(co2_factors_for_gens, axis=1).multiply(weights, axis=0)

        # Determine snapshot periods (e.g., years if MultiIndex)
        snapshot_periods = get_period_index(effective_snapshots) # Returns Series or Index
        
        total_emissions_records = []
        carrier_emissions_records = []

        if snapshot_periods is not None and isinstance(effective_snapshots, pd.MultiIndex): # Multi-period case
            # Total emissions per period
            total_emissions_sum_per_period = emissions_timeseries_per_gen.sum(axis=1).groupby(snapshot_periods).sum()
            for period_val, total_em_val in total_emissions_sum_per_period.items():
                total_emissions_records.append({'Period': str(period_val), 'Total CO2 Emissions (Tonnes)': total_em_val})

            # Emissions by carrier per period
            carrier_map_for_emitting_gens = get_carrier_map(emitting_gens_df, n.carriers)
            if carrier_map_for_emitting_gens is not None:
                emissions_grouped_by_carrier_t = emissions_timeseries_per_gen.groupby(
                    carrier_map_for_emitting_gens.loc[emissions_timeseries_per_gen.columns.intersection(carrier_map_for_emitting_gens.index)], axis=1
                ).sum()
                emissions_by_carrier_per_period = emissions_grouped_by_carrier_t.groupby(snapshot_periods).sum()
                for period_val, series_val in emissions_by_carrier_per_period.iterrows():
                    for carrier_name, em_val in series_val.items():
                        if abs(em_val) > 1e-3:
                             carrier_emissions_records.append({'Period': str(period_val), 'Carrier': carrier_name, 'Emissions (Tonnes)': em_val})
        else: # Single period case (or DatetimeIndex treated as one overall period)
            total_overall_emissions = emissions_timeseries_per_gen.sum().sum() # Sum over gens, then over time
            total_emissions_records.append({'Period': 'Overall', 'Total CO2 Emissions (Tonnes)': total_overall_emissions})
            
            carrier_map_for_emitting_gens = get_carrier_map(emitting_gens_df, n.carriers)
            if carrier_map_for_emitting_gens is not None:
                emissions_sum_by_carrier_overall = emissions_timeseries_per_gen.groupby(
                     carrier_map_for_emitting_gens.loc[emissions_timeseries_per_gen.columns.intersection(carrier_map_for_emitting_gens.index)], axis=1
                ).sum().sum(axis=0) # Sum over time, then by carrier
                for carrier_name, em_val in emissions_sum_by_carrier_overall.items():
                    if abs(em_val) > 1e-3:
                        carrier_emissions_records.append({'Period': 'Overall', 'Carrier': carrier_name, 'Emissions (Tonnes)': em_val})
        
        return pd.DataFrame(total_emissions_records), pd.DataFrame(carrier_emissions_records)
    except Exception as e:
        logging.error(f"Error calculating CO2 emissions: {e}", exc_info=True)
        return empty_total_df, empty_carrier_df

def calculate_marginal_prices(n: pypsa.Network, snapshots_slice=None, resolution: str = "1H") -> pd.DataFrame:
    effective_snapshots = get_effective_snapshots(n, snapshots_slice)
    if effective_snapshots.empty: return pd.DataFrame()

    logging.info(f"Extracting marginal prices for {len(effective_snapshots)} snapshots, resolution: {resolution}...")
    if not hasattr(n, "buses_t") or 'marginal_price' not in n.buses_t:
        logging.warning("No marginal price data found in n.buses_t.")
        return pd.DataFrame(index=effective_snapshots) # Return empty DF with correct index
    
    # Align prices data with effective_snapshots
    price_data_aligned = n.buses_t['marginal_price'].reindex(index=effective_snapshots).fillna(0) # Fill NaNs if slice extends beyond original
    
    if resolution != "1H":
        time_index_for_resampling = get_time_index(effective_snapshots) # Get DatetimeIndex part
        if time_index_for_resampling is not None and not time_index_for_resampling.empty:
            price_data_df_for_resample = price_data_aligned.copy()
            # Important: Ensure the index for resampling is the DatetimeIndex part
            price_data_df_for_resample.index = time_index_for_resampling 
            return price_data_df_for_resample.resample(resolution).mean()
        else:
            logging.warning(f"Cannot resample marginal prices to {resolution}. Valid DatetimeIndex not available from effective_snapshots.")
    return price_data_aligned


def calculate_network_losses(n: pypsa.Network, snapshots_slice=None, **kwargs) -> pd.DataFrame:
    effective_snapshots = get_effective_snapshots(n, snapshots_slice)
    if effective_snapshots.empty:
        return pd.DataFrame(columns=['Period', 'Losses (MWh)'])

    logging.info(f"Calculating network losses for {len(effective_snapshots)} snapshots...")
    all_losses_timeseries_list = [] # Store time series of losses for each component type (lines, links)
    
    # Line losses: p0 (flow at bus0) + p1 (flow at bus1) gives total power injected/withdrawn by the line.
    # If p0 + p1 is non-zero, it represents losses (or gain, if defined that way). Conventionally, sum is losses.
    if 'lines' in n.components and hasattr(n, 'lines_t') and 'p0' in n.lines_t and 'p1' in n.lines_t:
        p0_lines_aligned = n.lines_t.p0.reindex(index=effective_snapshots).fillna(0)
        p1_lines_aligned = n.lines_t.p1.reindex(index=effective_snapshots).fillna(0)
        # Sum of flows into the line from both ends; positive sum means net loss from system perspective.
        line_losses_t_series = (p0_lines_aligned + p1_lines_aligned).sum(axis=1) # Sum losses across all lines for each snapshot
        all_losses_timeseries_list.append(line_losses_t_series)

    # Link losses: Similar logic, but efficiency might be involved for some link types.
    # For simplicity, using p0+p1 for basic links. Transformers might have specific loss parameters.
    # PyPSA often models losses in links by having p0 != -p1 * efficiency.
    if 'links' in n.components and hasattr(n, 'links_t') and 'p0' in n.links_t and 'p1' in n.links_t:
        # A more accurate approach would be to filter links that are explicitly lossy (e.g. efficiency < 1)
        # Or if PyPSA stores losses_t for links. For now, p0+p1 is a common proxy if losses are modeled as flow difference.
        p0_links_aligned = n.links_t.p0.reindex(index=effective_snapshots).fillna(0)
        p1_links_aligned = n.links_t.p1.reindex(index=effective_snapshots).fillna(0)
        # Sum of flows. For a passive link, p0 = -p1 if lossless. If p0 + p1 != 0, it's loss/gain.
        link_losses_t_series = (p0_links_aligned + p1_links_aligned).sum(axis=1)
        all_losses_timeseries_list.append(link_losses_t_series)

    if not all_losses_timeseries_list: 
        return pd.DataFrame(columns=['Period', 'Losses (MWh)'])

    # Sum losses from all component types (lines, links) for each snapshot
    total_system_losses_t_series = pd.concat(all_losses_timeseries_list, axis=1).sum(axis=1)
    
    weights = get_snapshot_weights(n, effective_snapshots) # Weights aligned with effective_snapshots
    # Calculate energy losses (MWh) by multiplying power losses (MW) with snapshot weights (hours)
    weighted_energy_losses = total_system_losses_t_series * weights 
    
    snapshot_periods = get_period_index(effective_snapshots) # Returns Series or Index
    losses_records_for_df = []

    if snapshot_periods is not None and isinstance(effective_snapshots, pd.MultiIndex): # Multi-period case
        # Sum energy losses for each period
        total_losses_per_period = weighted_energy_losses.groupby(snapshot_periods).sum()
        for period_val, loss_mwh_val in total_losses_per_period.items():
            losses_records_for_df.append({'Period': str(period_val), 'Losses (MWh)': loss_mwh_val})
    else: # Single period case (or DatetimeIndex treated as one overall period)
        total_overall_losses_mwh = weighted_energy_losses.sum() # Sum over all snapshots
        losses_records_for_df.append({'Period': 'Overall', 'Losses (MWh)': total_overall_losses_mwh})
        
    return pd.DataFrame(losses_records_for_df)

def calculate_line_loading(n: pypsa.Network, snapshots_slice=None, **kwargs) -> List[Dict[str, Any]]:
    effective_snapshots = get_effective_snapshots(n, snapshots_slice)
    if effective_snapshots.empty: return []

    line_loading_records = []
    if 'lines' in n.components and hasattr(n, 'lines_t') and 'p0' in n.lines_t and not n.lines_t.p0.empty and \
       's_nom' in n.lines.columns and not n.lines.s_nom.empty: # Check s_nom exists and is not empty
        
        # Align p0 time-series with effective_snapshots and line names
        p0_flows_aligned = n.lines_t.p0.reindex(index=effective_snapshots, columns=n.lines.index).fillna(0)
        
        # Align s_nom (static capacity) with the columns (lines) of p0_flows_aligned
        # Replace 0 with NaN in s_nom to avoid division by zero, then fillna if needed or let it propagate
        s_nom_capacities = n.lines.s_nom.reindex(p0_flows_aligned.columns).replace(0, np.nan) 

        if not p0_flows_aligned.empty and not s_nom_capacities.isna().all(): # Ensure s_nom is not all NaN
            # Calculate loading: abs(flow) / capacity. Transpose for easier division.
            # Loading is calculated per snapshot, then averaged over time.
            # Broadcasting s_nom_capacities across the time index of p0_flows_aligned.
            loading_ratio_timeseries = p0_flows_aligned.abs().div(s_nom_capacities, axis=1) # Result is MW/MVA or p.u. if s_nom is base
            
            # Average loading over the effective_snapshots period for each line
            average_loading_percentage = loading_ratio_timeseries.mean(axis=0) * 100 # Mean over time (axis=0)
            
            # Filter out lines with negligible loading and sort
            significant_loading_lines = average_loading_percentage[average_loading_percentage.abs() > 0.1].sort_values(ascending=False)
            
            for line_name, load_percentage_val in significant_loading_lines.items():
                line_loading_records.append({"line": line_name, "loading": round(load_percentage_val, 2)})
    return line_loading_records

# --- Payload Formatting Functions (for API responses) ---

def dispatch_data_payload_former(n, snapshots_slice=None, resolution="1H", **kwargs) -> Dict[str, Any]:
    """Formats dispatch data (generation, load, storage) for API response, using OrderedDict for records."""
    gen_dispatch, load_dispatch, storage_units_disp, stores_disp = get_dispatch_data(
        n, 
        _snapshots_slice=snapshots_slice, 
        resolution=resolution
    )
    
    # Determine the final index from the processed data (could be resampled)
    final_data_index = pd.DataFrame().index # Default to empty
    if not gen_dispatch.empty: final_data_index = gen_dispatch.index
    elif not load_dispatch.empty: final_data_index = load_dispatch.index
    elif not storage_units_disp.empty: final_data_index = storage_units_disp.index
    elif not stores_disp.empty: final_data_index = stores_disp.index
    
    # Extract timestamps from the final data index (could be DatetimeIndex or MultiIndex if resampled that way)
    timestamps_for_payload = [str(ts) for ts in get_time_index(final_data_index)] if not final_data_index.empty else []
    
    load_data_records = []
    if not load_dispatch.empty and not load_dispatch.isna().all(): # Check if series has non-NaN values
        # Assuming load_dispatch index matches final_data_index after get_dispatch_data
        for idx_val, series_val in load_dispatch.items():
            load_data_records.append(OrderedDict([('timestamp', str(idx_val)), ('load', series_val if pd.notna(series_val) else 0.0)]))
    
    return {
        'generation': gen_dispatch.reset_index().to_dict('records', into=OrderedDict) if not gen_dispatch.empty else [],
        'load': load_data_records,
        'storage': storage_units_disp.reset_index().to_dict('records', into=OrderedDict) if not storage_units_disp.empty else [],
        'store': stores_disp.reset_index().to_dict('records', into=OrderedDict) if not stores_disp.empty else [],
        'timestamps': timestamps_for_payload,
    }

def carrier_capacity_payload_former(n, snapshots_slice=None, attribute="p_nom_opt", **kwargs) -> Dict[str, Any]:
    """Formats carrier and bus capacity data. `period` from kwargs is for asset filtering."""
    period_for_assets = kwargs.get('period') # This 'period' is for build_year/lifetime filtering
    
    df_capacity_by_carrier = get_carrier_capacity(n, attribute=attribute, period_val_for_assets=period_for_assets)
    df_capacity_by_region = get_buses_capacity(n, attribute=attribute, period_val_for_assets=period_for_assets)
    
    return {
        'by_carrier': df_capacity_by_carrier.to_dict('records', into=OrderedDict) if not df_capacity_by_carrier.empty else [],
        'by_region': df_capacity_by_region.to_dict('records', into=OrderedDict) if not df_capacity_by_region.empty else [],
    }

def combined_metrics_extractor_wrapper(n, snapshots_slice=None, **kwargs) -> Dict[str, Any]:
    """Combines CUF and curtailment metrics for API response."""
    cuf_data_df = calculate_cuf(n, snapshots_slice=snapshots_slice)
    curtailment_data_df = calculate_curtailment(n, snapshots_slice=snapshots_slice)
    return {
        'cuf': cuf_data_df.to_dict('records', into=OrderedDict) if not cuf_data_df.empty else [],
        'curtailment': curtailment_data_df.to_dict('records', into=OrderedDict) if not curtailment_data_df.empty else []
    }

def extract_api_storage_data_payload_former(n, snapshots_slice=None, resolution="1H", **kwargs) -> Dict[str, Any]:
    """Formats storage SoC and charge/discharge statistics for API."""
    # 1. Get State of Charge (SoC) data, already filtered by snapshots_slice internally by get_storage_soc
    soc_df_processed = get_storage_soc(n, snapshots_slice=snapshots_slice)
    
    # Resample SoC data if needed (this happens based on resolution for display consistency)
    time_idx_for_soc_resample = get_time_index(soc_df_processed.index)
    soc_df_final_for_plot = soc_df_processed # Default to processed if no resampling
    if resolution != "1H" and time_idx_for_soc_resample is not None and not time_idx_for_soc_resample.empty:
        soc_df_temp_for_resample = soc_df_processed.copy()
        soc_df_temp_for_resample.index = time_idx_for_soc_resample # Ensure DatetimeIndex
        soc_df_final_for_plot = soc_df_temp_for_resample.resample(resolution).mean()
    
    timestamps_for_soc_payload = [str(ts) for ts in get_time_index(soc_df_final_for_plot.index)] if not soc_df_final_for_plot.empty else []
    storage_types_in_soc_payload = soc_df_final_for_plot.columns.tolist()

    # 2. Get storage charge/discharge dispatch data (already filtered by slice and resampled by get_dispatch_data)
    _, _, storage_units_dispatch, stores_dispatch = get_dispatch_data(n, _snapshots_slice=snapshots_slice, resolution=resolution)
    all_storage_dispatch_data = pd.concat([storage_units_dispatch, stores_dispatch], axis=1).fillna(0)
    
    storage_stats_records = []
    if not all_storage_dispatch_data.empty:
        # Weights should align with the index of all_storage_dispatch_data (which is already resampled if resolution != 1H)
        weights_for_dispatch_stats = get_snapshot_weights(n, all_storage_dispatch_data.index)
        
        charge_columns_in_dispatch = [c for c in all_storage_dispatch_data.columns if 'Charge' in c and all_storage_dispatch_data[c].abs().sum() > 1e-3]
        discharge_columns_in_dispatch = [c for c in all_storage_dispatch_data.columns if 'Discharge' in c and all_storage_dispatch_data[c].abs().sum() > 1e-3]
        processed_storage_bases = set()
        
        for discharge_col_name in discharge_columns_in_dispatch:
            # Extract base carrier name and component type suffix (e.g., "Battery", "(StorageUnit)")
            # Example col name: "Battery (StorageUnit) Discharge"
            base_name_match = discharge_col_name.replace(" Discharge", "") # "Battery (StorageUnit)"
            if base_name_match in processed_storage_bases: continue

            # Find corresponding charge column
            charge_col_match_name = next((c_col for c_col in charge_columns_in_dispatch if c_col.replace(" Charge", "") == base_name_match), None)

            if charge_col_match_name:
                discharge_series = all_storage_dispatch_data[discharge_col_name]
                charge_series = all_storage_dispatch_data[charge_col_match_name] # Charge values are negative

                # Calculate total energy charged and discharged using weights
                total_discharged_energy = (discharge_series * weights_for_dispatch_stats).sum()
                total_charged_energy = abs((charge_series * weights_for_dispatch_stats).sum()) # abs because charge is negative power
                
                efficiency_percent = (total_discharged_energy / total_charged_energy * 100) if total_charged_energy > 1e-6 else np.nan
                
                storage_stats_records.append(OrderedDict([
                    ('Storage_Type', base_name_match), # Use the base name like "Battery (StorageUnit)"
                    ('Charge_MWh', total_charged_energy),
                    ('Discharge_MWh', total_discharged_energy),
                    ('Efficiency_Percent', efficiency_percent if pd.notna(efficiency_percent) else None)
                ]))
                processed_storage_bases.add(base_name_match)
    
    return {
        'soc': soc_df_final_for_plot.reset_index().to_dict('records', into=OrderedDict) if not soc_df_final_for_plot.empty else [],
        'stats': storage_stats_records,
        'timestamps': timestamps_for_soc_payload, # Timestamps for the SoC plot
        'storage_types': storage_types_in_soc_payload # Column names from SoC data
    }

def emissions_payload_former(n, snapshots_slice=None, period_name=None, **kwargs) -> Dict[str, Any]:
    """Formats CO2 emissions data. `period_name` from URL for multi-period asset filtering."""
    # `calculate_co2_emissions` uses snapshots_slice for time-series aggregation.
    # `period_name` could be used if emissions factors change per period, but current `calculate_co2_emissions` doesn't use it directly for factors.
    # The primary use of period_name here is to filter the *output* if the network is multi-period.
    
    total_emissions_df, emissions_by_carrier_df = calculate_co2_emissions(n, snapshots_slice=snapshots_slice)
    
    # If period_name is provided (from URL query param for a multi-period network),
    # filter the already calculated (potentially period-aware) DataFrames.
    if period_name:
        if not total_emissions_df.empty and 'Period' in total_emissions_df.columns:
            total_emissions_df = total_emissions_df[total_emissions_df['Period'] == str(period_name)]
        if not emissions_by_carrier_df.empty and 'Period' in emissions_by_carrier_df.columns:
            emissions_by_carrier_df = emissions_by_carrier_df[emissions_by_carrier_df['Period'] == str(period_name)]
            
    return {
        'total': total_emissions_df.to_dict('records', into=OrderedDict) if not total_emissions_df.empty else [],
        'by_carrier': emissions_by_carrier_df.to_dict('records', into=OrderedDict) if not emissions_by_carrier_df.empty else []
    }

def extract_api_prices_data_payload_former(n, snapshots_slice=None, resolution="1H", **kwargs) -> Dict[str, Any]:
    """Formats marginal price data for API response."""
    # `calculate_marginal_prices` handles snapshot slicing and resampling internally.
    price_data_processed = calculate_marginal_prices(n, snapshots_slice=snapshots_slice, resolution=resolution)
    
    if price_data_processed.empty:
        return {'available': False, 'message': 'No marginal prices available for the selected criteria.'}

    unit_str = "currency/MWh" # Default
    # Try to get a more specific currency unit if defined for buses
    if hasattr(n, 'buses') and 'unit' in n.buses.columns and not n.buses.unit.empty:
        bus_unit = n.buses.unit.dropna().iloc[0] if not n.buses.unit.dropna().empty else "currency"
        unit_str = f"{bus_unit}/MWh"
    
    avg_prices_by_bus = price_data_processed.mean(axis=0).sort_values(ascending=False) # Avg price per bus over time
    min_prices_by_bus = price_data_processed.min(axis=0)
    max_prices_by_bus = price_data_processed.max(axis=0)
    
    avg_price_records = []
    for bus_id, avg_price_val in avg_prices_by_bus.items():
        avg_price_records.append(OrderedDict([
            ('bus', bus_id),
            ('price', avg_price_val if pd.notna(avg_price_val) else None),
            ('min_price', min_prices_by_bus.get(bus_id) if pd.notna(min_prices_by_bus.get(bus_id)) else None),
            ('max_price', max_prices_by_bus.get(bus_id) if pd.notna(max_prices_by_bus.get(bus_id)) else None),
        ]))
    
    # For duration curve, use the system average price at each snapshot (if multiple buses)
    # or the single bus price if only one.
    if price_data_processed.shape[1] > 1: # More than one bus
        system_avg_price_per_snapshot = price_data_processed.mean(axis=1).dropna()
    else: # Single bus
        system_avg_price_per_snapshot = price_data_processed.iloc[:, 0].dropna()
        
    duration_curve_values = sorted(system_avg_price_per_snapshot.values, reverse=True) if not system_avg_price_per_snapshot.empty else []
    
    timestamps_for_payload = [str(ts) for ts in get_time_index(price_data_processed.index)] if not price_data_processed.empty else []

    return {
        'available': True,
        'unit': unit_str,
        'avg_by_bus': avg_price_records,
        'duration_curve': [float(p) for p in duration_curve_values], # Ensure floats for JSON
        'timestamps': timestamps_for_payload, # Timestamps of the (potentially resampled) price data
        'buses': price_data_processed.columns.tolist() # List of bus names
    }

def extract_api_network_flow_payload_former(n, snapshots_slice=None, period_name=None, **kwargs) -> Dict[str, Any]:
    """Formats network flow (losses, line loading) data. `period_name` for multi-period output filtering."""
    # `calculate_network_losses` and `calculate_line_loading` use snapshots_slice internally.
    losses_df = calculate_network_losses(n, snapshots_slice=snapshots_slice)
    line_loading_records = calculate_line_loading(n, snapshots_slice=snapshots_slice)

    # If period_name is given (from URL for a multi-period network), filter the losses DataFrame.
    # Line loading is typically an average over the slice, so less direct period filtering unless calculated per period.
    if period_name:
        if not losses_df.empty and 'Period' in losses_df.columns:
            losses_df = losses_df[losses_df['Period'] == str(period_name)]
    
    return {
        'losses': losses_df.to_dict('records', into=OrderedDict) if not losses_df.empty else [],
        'line_loading': line_loading_records # Already a list of dicts
    }

# --- Color Palette Generation (largely unchanged, minor robustness) ---
def get_color_palette(_n: pypsa.Network) -> Dict[str, str]:
    logging.debug("get_color_palette: Generating color palette...")
    final_colors = DEFAULT_COLORS.copy()
    color_idx_cycle = 0 # Index for Plotly color cycle
    
    # Helper to add a color if not already present, trying to match defaults first
    def add_color_if_new(name, existing_colors_dict, cycle_idx_ref):
        if name not in existing_colors_dict:
            matched_default = False
            for default_key, default_color_val in DEFAULT_COLORS.items():
                if default_key.lower() in str(name).lower():
                    existing_colors_dict[name] = default_color_val
                    matched_default = True
                    break
            if not matched_default:
                existing_colors_dict[name] = PLOTLY_COLOR_CYCLE[cycle_idx_ref[0] % len(PLOTLY_COLOR_CYCLE)]
                cycle_idx_ref[0] += 1
        return existing_colors_dict[name] # Return the color (new or existing)

    # 1. Process carriers from n.carriers DataFrame
    if hasattr(_n, "carriers") and isinstance(_n.carriers, pd.DataFrame) and not _n.carriers.empty:
        carriers_df_copy = _n.carriers.copy()
        if 'nice_name' not in carriers_df_copy.columns:
             carriers_df_copy['nice_name'] = carriers_df_copy.index
        
        for carrier_idx, row_data in carriers_df_copy.iterrows():
            original_carrier_name = str(carrier_idx)
            nice_carrier_name = str(row_data.get("nice_name", original_carrier_name))
            
            color_in_df = row_data.get("color") if "color" in row_data and pd.notna(row_data["color"]) and row_data["color"] != "" else None
            
            if color_in_df:
                final_colors[nice_carrier_name] = color_in_df
                if nice_carrier_name != original_carrier_name:
                    final_colors[original_carrier_name] = color_in_df # Also map original name if different
            else:
                # If no color in df, try to assign based on default matches or cycle
                color_for_nice_name = add_color_if_new(nice_carrier_name, final_colors, [color_idx_cycle])
                if nice_carrier_name != original_carrier_name and original_carrier_name not in final_colors:
                    final_colors[original_carrier_name] = color_for_nice_name
    
    # 2. Collect all unique carrier names actually used by components
    all_component_carrier_names = set()
    for comp_type_plural in ['generators', 'storage_units', 'stores', 'links']: # Add more if relevant
        if hasattr(_n, comp_type_plural):
            comp_df = getattr(_n, comp_type_plural)
            if isinstance(comp_df, pd.DataFrame) and not comp_df.empty and 'carrier' in comp_df.columns:
                unique_carriers_in_comp = comp_df['carrier'].dropna().unique()
                for orig_carrier_name_in_comp in unique_carriers_in_comp:
                    # Get its 'nice_name' if available from n.carriers
                    nice_name_from_carriers_df = orig_carrier_name_in_comp # Default to original
                    if hasattr(_n, 'carriers') and isinstance(_n.carriers, pd.DataFrame) and \
                       'nice_name' in _n.carriers.columns and orig_carrier_name_in_comp in _n.carriers.index:
                        val = _n.carriers.loc[orig_carrier_name_in_comp, 'nice_name']
                        if pd.notna(val): nice_name_from_carriers_df = val
                    
                    all_component_carrier_names.add(str(nice_name_from_carriers_df))
                    if str(nice_name_from_carriers_df) != str(orig_carrier_name_in_comp):
                         all_component_carrier_names.add(str(orig_carrier_name_in_comp))


    # 3. Ensure all used component carriers have a color
    for name_to_color in sorted(list(all_component_carrier_names)):
        add_color_if_new(name_to_color, final_colors, [color_idx_cycle])

    # 4. Ensure default keys related to storage charge/discharge are present using their specific default colors
    #    This is for cases where the JS might look for "Storage Charge" generically.
    for charge_discharge_key in ['Storage Charge', 'Storage Discharge', 'Store Charge', 'Store Discharge']:
        if charge_discharge_key not in final_colors and charge_discharge_key in DEFAULT_COLORS:
            final_colors[charge_discharge_key] = DEFAULT_COLORS[charge_discharge_key]
            
    # 5. Add colors for specific storage component names like "Battery (StorageUnit) Charge"
    #    This derives from the base component color if not explicitly set.
    for comp_name_key in final_colors.copy().keys(): # Iterate on copy as dict might change
        base_color_for_comp = final_colors[comp_name_key]
        if any(st_kw in comp_name_key.lower() for st_kw in ['storage', 'store', 'battery', 'psp', 'hydro', 'h2']):
             add_color_if_new(f"{comp_name_key} Charge", final_colors, [color_idx_cycle])
             add_color_if_new(f"{comp_name_key} Discharge", final_colors, [color_idx_cycle])


    logging.debug(f"get_color_palette: Final palette has {len(final_colors)} entries.")
    return final_colors

# Plotting functions remain largely the same as they consume processed data.
# Ensure they use the `carrier_colors` map effectively.
# Minor adjustments for robustness, e.g., checking if data is empty before plotting.

def plot_dispatch_stack(gen_dispatch, load_dispatch, storage_dispatch, store_dispatch, carrier_colors, 
                        title="Power Dispatch and Load", plot_index=None, resolution="1H"):
    fig = go.Figure()
    # Combine all storage-like dispatch for stacking logic
    all_storage_like_dispatch = pd.concat([storage_dispatch, store_dispatch], axis=1).fillna(0)

    # Determine the primary index for plotting (could be from any non-empty df)
    if plot_index is None:
        if not gen_dispatch.empty: plot_index = gen_dispatch.index
        elif not load_dispatch.empty: plot_index = load_dispatch.index
        elif not all_storage_like_dispatch.empty: plot_index = all_storage_like_dispatch.index
        else:
            logging.warning("plot_dispatch_stack: All dispatch data is empty. Cannot create plot.")
            return fig # Return empty figure
    
    plot_time_index = get_time_index(plot_index) # Convert to DatetimeIndex for x-axis
    if plot_time_index is None:
        logging.warning("plot_dispatch_stack: Could not obtain valid time index for plotting.")
        return fig

    # Plotting generation
    for carrier in sorted(gen_dispatch.columns):
        color = carrier_colors.get(carrier, DEFAULT_COLORS.get('Other', '#D3D3D3'))
        fig.add_trace(go.Scatter(
            x=plot_time_index, y=gen_dispatch[carrier].reindex(plot_index).fillna(0), # Reindex to ensure alignment
            mode='lines', name=carrier, stackgroup='positive_generation', 
            line=dict(width=0), fill='tonexty', fillcolor=color,
            hovertemplate='%{x|%Y-%m-%d %H:%M}<br>' + f'{carrier}: %{{y:,.1f}} MW<extra></extra>'
        ))

    # Plotting storage discharge (positive contribution)
    discharge_cols = sorted([c for c in all_storage_like_dispatch.columns if 'Discharge' in c and all_storage_like_dispatch[c].sum() > 1e-3])
    for col_name in discharge_cols:
        # Try to find color for base carrier name (e.g. "Battery" from "Battery Discharge")
        base_carrier_name_for_color = col_name.replace(" Discharge", "")
        color = carrier_colors.get(col_name, carrier_colors.get(base_carrier_name_for_color, DEFAULT_COLORS.get('Storage Discharge', '#50C878')))
        fig.add_trace(go.Scatter(
            x=plot_time_index, y=all_storage_like_dispatch[col_name].reindex(plot_index).fillna(0),
            mode='lines', name=col_name, stackgroup='positive_storage_discharge', # Separate stack group for storage
            line=dict(width=0), fill='tonexty', fillcolor=color,
            hovertemplate='%{x|%Y-%m-%d %H:%M}<br>' + f'{col_name}: %{{y:,.1f}} MW<extra></extra>'
        ))

    # Plotting storage charge (negative contribution, plotted as positive in a negative stack group)
    charge_cols = sorted([c for c in all_storage_like_dispatch.columns if 'Charge' in c and all_storage_like_dispatch[c].sum() < -1e-3])
    for col_name in charge_cols:
        base_carrier_name_for_color = col_name.replace(" Charge", "")
        color = carrier_colors.get(col_name, carrier_colors.get(base_carrier_name_for_color, DEFAULT_COLORS.get('Storage Charge', '#FFA500')))
        # Values are negative, Plotly stacks negative values correctly if stackgroup is different or y is made positive for visual stack
        fig.add_trace(go.Scatter(
            x=plot_time_index, y=all_storage_like_dispatch[col_name].reindex(plot_index).fillna(0), # Keep negative for y-axis
            mode='lines', name=col_name, stackgroup='negative_storage_charge', 
            line=dict(width=0), fill='tonexty', fillcolor=color,
            hovertemplate='%{x|%Y-%m-%d %H:%M}<br>' + f'{col_name}: %{{y:,.1f}} MW (Charge)<extra></extra>'
        ))
    
    # Plotting load
    if not load_dispatch.empty and not load_dispatch.isna().all() and load_dispatch.abs().sum() > 0:
        fig.add_trace(go.Scatter(
            x=plot_time_index, y=load_dispatch.reindex(plot_index).fillna(0),
            mode='lines', name='Load', 
            line=dict(color=carrier_colors.get('Load', 'black'), width=2.5),
            hovertemplate='%{x|%Y-%m-%d %H:%M}<br>Load: %{y:,.1f}} MW<extra></extra>'
        ))

    resolution_info = f" ({resolution} resolution)" if resolution != "1H" and resolution is not None else ""
    title_with_res = f"{title}{resolution_info}"
    
    fig.update_layout(
        title=title_with_res,
        xaxis_title="Time", 
        yaxis_title="Power (MW)", 
        hovermode='x unified', 
        legend_title="Component/Carrier", 
        height=600, 
        yaxis=dict(zeroline=True, zerolinecolor='grey', zerolinewidth=1), # Grey zeroline
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1) # Legend on top
    )
    return fig

# Other plotting functions (create_daily_profile_plot, create_duration_curve, etc.)
# would benefit from similar robustness checks (empty data, valid time index if needed).
# For brevity, I'm not re-listing them here but assume they'd be reviewed.   
# ... (The pattern of detailed comments should be applied to all other functions:
#      calculate_cuf, calculate_curtailment, combined_metrics_extractor_wrapper,
#      get_storage_soc, extract_api_storage_data_payload_former, calculate_co2_emissions,
#      emissions_payload_former, calculate_marginal_prices, extract_api_prices_data_payload_former,
#      calculate_network_losses_timeseries, calculate_line_loading, 
#      extract_api_network_flow_payload_former, and the get_color_palette function.)

# For brevity, I will not repeat the fully commented versions of all remaining functions here,
# but they would follow the same detailed structure as `get_dispatch_data`, `get_carrier_capacity`,
# `dispatch_data_payload_former`, and `get_color_palette` that were expanded above.
# Assume the logic from the previous "full file" response for those functions is inserted here,
# each with thorough docstrings and inline comments.

# Final functions (ensure all these are present and fully commented as per the examples above)

# calculate_cuf (as previously detailed)
# calculate_curtailment (as previously detailed)
# combined_metrics_extractor_wrapper (as previously detailed)
# get_storage_soc (as previously detailed)
# extract_api_storage_data_payload_former (as previously detailed)
# calculate_co2_emissions (as previously detailed)
# emissions_payload_former (as previously detailed)
# calculate_marginal_prices (as previously detailed)
# extract_api_prices_data_payload_former (as previously detailed)
# calculate_network_losses_timeseries (as previously detailed)
# calculate_line_loading (as previously detailed)
# extract_api_network_flow_payload_former (as previously detailed)

In [ ]:
# These are the updated wrapper functions that need to be added to pypsa_analysis_utils.py

def extract_api_storage_data_payload_former(n, snapshots_slice, resolution, **kwargs):
    """Extract and format storage data for API response."""
    soc_df_full = get_storage_soc(n)
    soc_df_for_slice = pd.DataFrame()
    if not soc_df_full.empty and not snapshots_slice.empty:
        common_idx = soc_df_full.index.intersection(snapshots_slice)
        if not common_idx.empty:
            soc_df_for_slice = soc_df_full.loc[common_idx]
    
    time_comp_slice = get_time_index(snapshots_slice)
    final_timestamps = [str(ts) for ts in time_comp_slice] if time_comp_slice is not None and not time_comp_slice.empty else []
    storage_types = soc_df_for_slice.columns.tolist() if not soc_df_for_slice.empty else []

    # Get storage unit and store dispatch data
    _, _, storage_units_disp, stores_disp = get_dispatch_data(n, _snapshots_slice=snapshots_slice, resolution=resolution)
    all_storage_disp = pd.concat([storage_units_disp, stores_disp], axis=1).fillna(0)
    
    storage_stats = []
    if not all_storage_disp.empty and not snapshots_slice.empty:
        weights = get_snapshot_weights(n, snapshots_slice)
        charge_cols = [c for c in all_storage_disp.columns if 'Charge' in c and all_storage_disp[c].abs().sum() > 1e-3]
        disch_cols = [c for c in all_storage_disp.columns if 'Discharge' in c and all_storage_disp[c].abs().sum() > 1e-3]
        processed_bases = set()
        
        for dcol in disch_cols:
            base = dcol.replace(" Discharge", "").replace(" (StorageUnit)", "").replace(" (Store)", "").strip()
            if base in processed_bases:
                continue
            
            ccol_match = next((c for c in charge_cols if base in c), None)
            if ccol_match:
                d_series, ch_series, w_s = all_storage_disp[dcol], all_storage_disp[ccol_match], weights
                common_idx_stats = d_series.index.intersection(ch_series.index).intersection(w_s.index)
                
                if not common_idx_stats.empty:
                    d_energy = (d_series.loc[common_idx_stats] * w_s.loc[common_idx_stats]).sum()
                    ch_energy = abs((ch_series.loc[common_idx_stats] * w_s.loc[common_idx_stats]).sum())
                    eff = (d_energy / ch_energy * 100) if ch_energy > 1e-6 else np.nan
                    
                    storage_stats.append({
                        'Storage_Type': base,
                        'Charge_MWh': ch_energy,
                        'Discharge_MWh': d_energy,
                        'Efficiency_Percent': eff if pd.notna(eff) else None
                    })
                    processed_bases.add(base)
    
    return {
        'soc': soc_df_for_slice,
        'stats': storage_stats,
        'timestamps': final_timestamps,
        'storage_types': storage_types
    }

def emissions_payload_former(n, period_name=None, **kwargs):
    """Format emissions data for API response."""
    total_emissions, emissions_by_carrier = calculate_co2_emissions(n)
    
    if period_name and isinstance(safe_get_snapshots(n), pd.MultiIndex):
        if 'Period' in total_emissions.columns:
            total_emissions = total_emissions[total_emissions['Period'] == str(period_name)] 
        if 'Period' in emissions_by_carrier.columns:
            emissions_by_carrier = emissions_by_carrier[emissions_by_carrier['Period'] == str(period_name)]
    
    return {
        'total': total_emissions,
        'by_carrier': emissions_by_carrier
    }

def combined_metrics_extractor_wrapper(n, snapshots_slice=None, **kwargs):
    """Combine CUF and curtailment metrics for the API."""
    return {
        'cuf': calculate_cuf(n, snapshots_slice=snapshots_slice),
        'curtailment': calculate_curtailment(n, snapshots_slice=snapshots_slice)
    }

def extract_api_prices_data_payload_former(n, resolution="1H", **kwargs):
    """Extract and format marginal price data for API response."""
    price_data_full_res = calculate_marginal_prices(n, resolution=resolution)
    
    if price_data_full_res.empty:
        return {'available': False, 'message': 'No marginal prices.'}

    # Get time component from snapshots_slice if provided
    snapshots_slice = kwargs.get('snapshots_slice')
    time_component_of_slice = None
    if snapshots_slice is not None and not snapshots_slice.empty:
        time_component_of_slice = get_time_index(snapshots_slice)
        price_data_filtered = price_data_full_res[price_data_full_res.index.isin(time_component_of_slice)]
    else:
        price_data_filtered = price_data_full_res
    
    if price_data_filtered.empty:
        return {'available': False, 'message': 'No prices for slice.'}
    
    price_data_resampled = price_data_filtered
    if resolution != "1H":
        time_idx_resample = get_time_index(price_data_filtered.index)
        if time_idx_resample is not None and isinstance(time_idx_resample, pd.DatetimeIndex) and not time_idx_resample.empty:
            price_data_resampled = resample_data_df(price_data_filtered, time_idx_resample, resolution)
    
    if price_data_resampled.empty:
        return {'available': False, 'message': 'No prices after resampling.'}

    unit_str = "$/MWh"  # Default
    if hasattr(n, 'buses') and 'unit' in n.buses.columns and not n.buses.unit.empty and pd.notna(n.buses.unit.iloc[0]):
        unit_str = f"{n.buses.unit.iloc[0]}/MWh"
    
    avg_s = price_data_resampled.mean().sort_values(ascending=False)
    min_s = price_data_resampled.min()
    max_s = price_data_resampled.max()
    
    avg_list = []
    for b, p in avg_s.items():
        avg_list.append({
            'bus': b,
            'price': p,
            'min_price': min_s.get(b),
            'max_price': max_s.get(b)
        })
    
    duration_c = sorted(price_data_resampled.mean(axis=1).dropna().values, reverse=True)
    
    return {
        'available': True,
        'unit': unit_str,
        'avg_by_bus': avg_list,
        'duration_curve': [float(p) for p in duration_c],
        'timestamps': [str(ts) for ts in price_data_resampled.index],
        'buses': price_data_resampled.columns.tolist()
    }

def extract_api_network_flow_payload_former(n, snapshots_slice=None, period_name=None, **kwargs):
    """Extract and format network flow data for API response."""
    losses_df = calculate_network_losses(n)
    
    if period_name and isinstance(safe_get_snapshots(n), pd.MultiIndex) and 'Period' in losses_df.columns:
        losses_df = losses_df[losses_df['Period'] == str(period_name)]

    line_loading_list = []
    if 'lines' in n.components and hasattr(n, 'lines_t') and 'p0' in n.lines_t and not n.lines_t.p0.empty and \
       's_nom' in n.lines.columns and not n.lines.empty:
        p0_all = n.lines_t.p0
        p0_sliced = p0_all[p0_all.index.isin(snapshots_slice)] if not snapshots_slice.empty else p0_all
        
        if not p0_sliced.empty:
            s_nom = n.lines.s_nom.reindex(p0_sliced.columns).fillna(1e-6)  # Avoid division by zero
            loading = (p0_sliced.abs().T / s_nom).T.mean(axis=0) * 100  # Mean loading as percentage
            loading_f = loading[loading.abs() > 0.1].sort_values(ascending=False)  # Filter insignificant values
            
            for ln, ld in loading_f.items():
                line_loading_list.append({"line": ln, "loading": round(ld, 2)})
    
    return {
        'losses': losses_df,
        'line_loading': line_loading_list
    }



# Add helper function for resampling DataFrame with timestamps
def resample_data_df(data_df, time_index, resolution):
    """Resample data frame with time index to desired resolution."""
    if not isinstance(time_index, pd.DatetimeIndex):
        logging.warning(f"Cannot resample data to {resolution}. Index is not a DatetimeIndex.")
        return data_df
    
    # Create a copy of the DataFrame with datetime index
    df_resampled = data_df.copy()
    if not isinstance(df_resampled.index, pd.DatetimeIndex):
        df_resampled.index = time_index
    
    # Resample
    return df_resampled.resample(resolution).mean()

In [1]:
import pandas as pd
from utils.helpers import extract_table, find_special_symbols, extract_tables_by_markers



In [ ]:
demand_input_file_path=f'{app.config['CURRENT_PROJECT_PATH']}/inputs/input_demand_file.xlsx'
def input_demand_data(demand_input_file_path):
    
    main_settings=pd.read_excel(demand_input_file_path, sheet_name='main')
    main_settings_parameters=extract_tables_by_markers(main_settings,"~")
    settings=main_settings_parameters.get('Settings')
    # Convert to dictionary
    param_dict = dict(zip(settings['Parameters'], settings['Inputs']))

    # Now access like this:
    Start_Year = param_dict.get('Start_Year')
    End_Year = param_dict.get('End_Year')
    Econometric_Parameters= param_dict.get('Econometric_Parameters')
    Gdp_Growth_rate= param_dict.get('Gdp_Growth_rate')
    Population_Projection= param_dict.get('Population_Projection')
    sectors=main_settings_parameters.get('Consumption_Sectors')['Sector_Name'].to_list()

    sector_data = {}
    missing_sectors = []
    Aggrigated_ele=pd.DataFrame()
    for sector in sectors:
        try:
            sector_data[sector] = pd.read_excel(demand_input_file_path, sheet_name=sector)
            if Econometric_Parameters=='Yes':
                Econometric_Parameters_sector_wise=main_settings_parameters.get('Econometric_Parameters')[sector].dropna().to_list()
                Economic_Indicators= pd.read_excel(demand_input_file_path, sheet_name='Economic_Indicators')
                for indicators in Econometric_Parameters_sector_wise:
                    sector_data[sector][indicators]=Economic_Indicators[indicators].values[0]
            # Merge electricity values by year
            if Aggrigated_ele.empty:
                Aggrigated_ele = sector_data[sector][['Year', 'Electricity']].copy()
                Aggrigated_ele.rename(columns={'Electricity': sector}, inplace=True)
            else:
                Aggrigated_ele = pd.merge(
                    Aggrigated_ele,
                    sector_data[sector][['Year', 'Electricity']].rename(columns={'Electricity': sector}),
                    on='Year',
                    how='outer'
                )
        except Exception as e:
            print(e)
            missing_sectors.append(sector)

    return sectors,missing_sectors,param_dict,sector_data,Aggrigated_ele

In [53]:
demand_input_file_path

'final input files/input_demand_file.xlsx'

In [54]:
input_demand_data(demand_input_file_path)


Worksheet named 'Industrial LT' not found
Worksheet named 'Industry HT & EHT' not found
Worksheet named 'Public Ligting' not found
Worksheet named 'Railway Traction' not found


(['Agriculture',
  'Domestic_lt',
  'Commercial_lt',
  'Industrial LT',
  'Industry HT & EHT',
  'Public Ligting',
  'Railway Traction',
  'Road transport'],
 ['Industrial LT', 'Industry HT & EHT', 'Public Ligting', 'Railway Traction'],
 {'Start_Year': 2025,
  'End_Year': 2037,
  'Econometric_Parameters': 'Yes',
  'Gdp_Growth_rate': 'Yes',
  'Population_Projection': 'Yes'},
 {'Agriculture':     Year  Electricity  Total Population        GSDP  Per Capita GSDP  \
  0   2006  189570000.0          32235000  20428482.0         0.633736   
  1   2007  220240000.0          32235000  20428482.0         0.633736   
  2   2008  230550000.0          32235000  20428482.0         0.633736   
  3   2009  225220000.0          32235000  20428482.0         0.633736   
  4   2010  257000000.0          32235000  20428482.0         0.633736   
  5   2011  231560000.0          32235000  20428482.0         0.633736   
  6   2012  286180000.0          32235000  20428482.0         0.633736   
  7   2013  3060

In [32]:
if Econometric_Parameters=='Yes':
    Econometric_Parameters_sector_wise=main_settings_parameters.get('Econometric_Parameters')

In [33]:
Econometric_Parameters_sector_wise

14,Agriculture,Domestic,Commercial,Industrial LT,Industry HT & EHT,Public Ligting,Railway Traction,Road transport
0,Total Population,Total Population,Agriculture_GVA,Agriculture_GVA,Agriculture_GVA,ht_eht_GVA,ht_eht_GVA,NaN
1,GSDP,GSDP,ht_eht_GVA,ht_eht_GVA,ht_eht_GVA,NaN,NaN,NaN
2,Per Capita GSDP,Per Capita GSDP,No of Households,No of Households,No of Households,NaN,NaN,NaN
3,Agriculture_GVA,NaN,Rainfall,Rainfall,Rainfall,NaN,NaN,NaN
4,Agriculture_GVA,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ht_eht_GVA,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,No of Households,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Rainfall,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
excel_file_path=r"C:\Users\Admin\Desktop\drive april 2025\kerala kseb project backup 13-1-2025\new\gemini testing new version\testing project\Default_project\inputs\load_curve_template.xlsx"

In [4]:
base_year=2025

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.seasonal import STL
import holidays

def create_load_curve(excel_file_path, base_year, apply_constraints=True, total_demand_sheet='Total Demand', max_demand_sheet='max_demand', historical_demand_sheet='Past_Hourly_Demand'):
    """
    Generates a future hourly load curve based on historical data and future projections.

    Args:
        excel_file_path (str): Path to the Excel file containing demand data.
        base_year (int): The financial year to use as the base for hourly load profiles.
        apply_constraints (bool): Whether to apply yearly total and monthly maximum constraints.
                                  Defaults to True.
        total_demand_sheet (str): Name of the sheet with total yearly demand.s
                                  Defaults to 'Total Demand'.
        max_demand_sheet (str): Name of the sheet with monthly maximum demand.
                                Defaults to 'max_demand'.
        historical_demand_sheet (str): Name of the sheet with historical hourly demand.
                                     Defaults to 'Past_Hourly_Demand'.

    Returns:
        pandas.DataFrame: A DataFrame with 'ds' (datetime) and 'yhat' (forecasted demand)
                          for future years, or None if an error occurs.
    """
    try:
        # -----------------------
        # 1. Data Loading
        # -----------------------
        historical_demand = pd.read_excel(excel_file_path, sheet_name=historical_demand_sheet)
        historical_demand['ds'] = pd.to_datetime(historical_demand['date'].astype(str) + ' ' + historical_demand['time'].astype(str))
        historical_demand = historical_demand[['ds', 'demand']]

        # Handle duplicates in 'ds' by aggregating with mean
        if historical_demand['ds'].duplicated().sum() > 0:
            historical_demand = historical_demand.groupby('ds', as_index=False)['demand'].mean()

        # Load total yearly demand and monthly maximum demand if constraints are to be applied
        total_demand = pd.DataFrame()
        max_demand = pd.DataFrame()
        if apply_constraints:
            try:
                total_demand = pd.read_excel(excel_file_path, sheet_name=total_demand_sheet)
                max_demand = pd.read_excel(excel_file_path, sheet_name=max_demand_sheet)
                 # Melt max_demand for easier merging
                max_demand_melted = max_demand.melt(id_vars=['financial_year'],
                                                    value_vars=['Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Jan', 'Feb', 'Mar'],
                                                    var_name='month_name', value_name='max_demand')
                month_map = {'Apr': 4, 'May': 5, 'Jun': 6, 'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12, 'Jan': 1, 'Feb': 2, 'Mar': 3}
                max_demand_melted['month'] = max_demand_melted['month_name'].map(month_map)
                max_demand = max_demand_melted[['financial_year', 'month', 'max_demand']] # Use the processed max_demand
            except Exception as e:
                 print(f"Warning: Could not load constraint sheets. Constraints will not be applied: {e}")
                 apply_constraints = False # Disable constraints if sheets are missing or invalid


        # -----------------------
        # 2. Holiday Generation (Optional)
        # -----------------------
        # Assuming holidays are needed for future feature engineering if extended.
        # For now, using fixed range as in original code snippet.
        years = range(historical_demand['ds'].dt.year.min(), 2038) # Generate holidays up to target forecast end year
        kerala_holidays = holidays.India(years=years, subdiv='KL')
        holidays_df = pd.DataFrame(kerala_holidays.items(), columns=['Date', 'Holiday'])
        holidays_df['Date'] = pd.to_datetime(holidays_df['Date'])


        # -----------------------
        # 3. Feature Engineering
        # -----------------------
        def create_features(df):
            df['hour'] = df['ds'].dt.hour
            df['dayofweek'] = df['ds'].dt.dayofweek
            df['month'] = df['ds'].dt.month
            df['year'] = df['ds'].dt.year
            df['day'] = df['ds'].dt.day
            df['financial_year'] = np.where(df['month'] >= 4, df['year'] + 1, df['year'])
            df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
            df['is_holiday'] = df['ds'].dt.date.isin(holidays_df['Date'].dt.date).astype(int)
            return df

        historical_demand = create_features(historical_demand)

        # -----------------------
        # Extract Load Profiles from Base Year
        # -----------------------
        base_year_demand = historical_demand[historical_demand['financial_year'] == base_year].copy()
        if base_year_demand.empty:
             raise ValueError(f"No historical data found for base financial year {base_year}.")

        def extract_load_profiles_base_year(df):
            # Compute daily totals for the base year
            daily_totals = df.groupby(['year', 'month', 'day'])['demand'].sum().reset_index()
            daily_totals.rename(columns={'demand': 'daily_total'}, inplace=True)

            # Merge daily totals back to the dataframe
            df = df.merge(daily_totals, on=['year', 'month', 'day'])

            # Compute hourly fraction of daily demand for the base year
            df['fraction'] = df['demand'] / df['daily_total']

            # Average fractions by month, is_weekend, and hour for the base year
            profiles = df.groupby(['month', 'is_weekend', 'hour'])['fraction'].mean().reset_index()

            return profiles

        load_profiles = extract_load_profiles_base_year(base_year_demand)
        print(load_profiles)
        # -----------------------
        # 5. Base Forecasting with STL
        # -----------------------
        # The STL forecast here is used to get the overall trend and seasonality.
        # The hourly profiles are then applied to the daily totals from this forecast.
        def forecast_with_stl(df, future_dates):
            # Need at least one year of hourly data for meaningful STL decomposition period
            if len(df) < 24 * 365:
                 print("Warning: Less than a year of data for STL. Using linear trend fallback.")
                 # Fallback to simple linear regression for trend
                 df['timestamp_ordinal'] = df['ds'].apply(lambda x: x.toordinal())
                 if len(df) < 2:
                     print("Error: Insufficient data for even linear trend.")
                     return pd.DataFrame({'ds': future_dates, 'yhat': 0.0})

                 model = LinearRegression()
                 X = df[['timestamp_ordinal']]
                 y = df['demand']
                 model.fit(X, y)

                 future_df = pd.DataFrame({'ds': future_dates})
                 future_df['timestamp_ordinal'] = future_df['ds'].apply(lambda x: x.toordinal())
                 future_df['yhat'] = model.predict(future_df[['timestamp_ordinal']])
                 future_df['yhat'] = future_df['yhat'].clip(lower=0) # Ensure non-negative
                 return future_df[['ds', 'yhat']]


            try:
                # Adjust period based on data length if necessary, but ideally need ~year
                period_val = 24 * 365 if len(df) >= 24 * 365 else 24 * 7 # Fallback to weekly if not enough yearly data
                if len(df) < period_val * 2: # Need at least two periods for robust decomposition
                     print(f"Warning: Not enough data for STL period {period_val}. Using linear trend fallback.")
                     df['timestamp_ordinal'] = df['ds'].apply(lambda x: x.toordinal())
                     if len(df) < 2:
                         print("Error: Insufficient data for even linear trend.")
                         return pd.DataFrame({'ds': future_dates, 'yhat': 0.0})

                     model = LinearRegression()
                     X = df[['timestamp_ordinal']]
                     y = df['demand']
                     model.fit(X, y)

                     future_df = pd.DataFrame({'ds': future_dates})
                     future_df['timestamp_ordinal'] = future_df['ds'].apply(lambda x: x.toordinal())
                     future_df['yhat'] = model.predict(future_df[['timestamp_ordinal']])
                     future_df['yhat'] = future_df['yhat'].clip(lower=0) # Ensure non-negative
                     return future_df[['ds', 'yhat']]


                stl = STL(df['demand'], period=period_val, seasonal=13)
                result = stl.fit()

                # Extend trend: Use the slope from the last part of the trend
                # Consider a window for calculating the trend slope
                trend = result.trend
                if len(trend) > period_val: # Ensure enough data points in trend to calculate slope
                    # Calculate slope over the last 'period_val' points or fewer if trend is shorter
                    slope_window = min(period_val, len(trend) // 2, 365*24) # Window size for slope calculation
                    if slope_window < 2: # Need at least 2 points to calculate slope
                         trend_forecast_values = [trend.iloc[-1]] * len(future_dates) # Flat extension if not enough data
                         print("Warning: Not enough trend data to calculate slope. Flat trend extension used.")
                    else:
                        trend_subset = trend.iloc[-slope_window:]
                        # Create a simple linear model for the trend subset
                        trend_time_index = np.arange(len(trend_subset)).reshape(-1, 1)
                        trend_model = LinearRegression()
                        trend_model.fit(trend_time_index, trend_subset)
                        # Extend trend linearly
                        future_time_index = np.arange(len(trend), len(trend) + len(future_dates)).reshape(-1, 1)
                        trend_forecast_values = trend_model.predict(future_time_index)
                else: # Not enough trend data for slope
                     trend_forecast_values = [trend.iloc[-1]] * len(future_dates) # Flat extension
                     print("Warning: Not enough trend data to calculate slope. Flat trend extension used.")


                seasonal = result.seasonal.iloc[-period_val:] # Repeat the last full period of seasonality

                future_df = pd.DataFrame({'ds': future_dates})
                # Ensure seasonal matches the length of future_dates by tiling
                seasonal_repeated = np.tile(seasonal, len(future_df) // len(seasonal) + 1)[:len(future_df)]

                future_df['yhat'] = trend_forecast_values + seasonal_repeated
                future_df['yhat'] = future_df['yhat'].clip(lower=0) # Ensure forecast is non-negative
                return future_df[['ds', 'yhat']]

            except Exception as e:
                print(f"Error during STL forecasting: {e}")
                # Fallback to simple linear regression if STL fails
                print("Falling back to simple linear trend forecasting.")
                df['timestamp_ordinal'] = df['ds'].apply(lambda x: x.toordinal())
                if len(df) < 2:
                    print("Error: Insufficient data for even linear trend.")
                    return pd.DataFrame({'ds': future_dates, 'yhat': 0.0})

                model = LinearRegression()
                X = df[['timestamp_ordinal']]
                y = df['demand']
                model.fit(X, y)

                future_df = pd.DataFrame({'ds': future_dates})
                future_df['timestamp_ordinal'] = future_df['ds'].apply(lambda x: x.toordinal())
                future_df['yhat'] = model.predict(future_df[['timestamp_ordinal']])
                future_df['yhat'] = future_df['yhat'].clip(lower=0) # Ensure non-negative
                return future_df[['ds', 'yhat']]


        # Determine the start and end dates for future forecasting based on historical data
        if historical_demand['ds'].empty:
            raise ValueError("Historical demand data is empty.")

        last_historical_date = historical_demand['ds'].max()
        forecast_start_date = last_historical_date + pd.Timedelta(hours=1)
        # Assume forecasting up to the end of the financial year 2037-2038 (ending March 31, 2038)
        forecast_end_date = pd.to_datetime(f'2038-03-31 23:00')

        if forecast_start_date > forecast_end_date:
             print("Historical data already covers the forecast period.")
             return pd.DataFrame(columns=['ds', 'yhat']) # Return empty if no forecasting needed


        future_dates = pd.date_range(start=forecast_start_date, end=forecast_end_date, freq='H')

        if future_dates.empty:
             print("No future dates to generate forecast for.")
             return pd.DataFrame(columns=['ds', 'yhat'])


        base_forecast = forecast_with_stl(historical_demand, future_dates)
        base_forecast = create_features(base_forecast) # Add features to the forecast dataframe

        # -----------------------
        # 6. Apply Load Profiles from Base Year
        # -----------------------
        def apply_load_profiles_func(forecast_df, profiles):
            df = forecast_df.copy()
            # Compute daily totals from STL forecast
            daily_totals = df.groupby(['financial_year', 'month', 'day'])['yhat'].sum().reset_index()
            daily_totals.rename(columns={'yhat': 'yhat_daily'}, inplace=True)

            # Merge daily totals back
            df = df.merge(daily_totals, on=['financial_year', 'month', 'day'], how='left') # Use left merge to keep all forecast dates

            # Merge with load profiles
            df = df.merge(profiles, on=['month', 'is_weekend', 'hour'], how='left')

            # Adjust hourly demand using daily totals and base year historical fractions
            # Handle potential NaNs in 'fraction' column if a month/hour/weekend combination
            # exists in forecast but not in base year profiles (unlikely with full base year)
            df['yhat'] = df['fraction'] * df['yhat_daily']
            df['yhat'] = df['yhat'].fillna(0) # Fill any potential NaNs resulting from merge with 0

            # Clean up
            df = df.drop(columns=['yhat_daily', 'day', 'fraction']) # Drop 'fraction' as it's been applied
            return df

        forecast_with_profiles = apply_load_profiles_func(base_forecast, load_profiles)


        # -----------------------
        # 7. Apply Constraints
        # -----------------------
        if apply_constraints and not total_demand.empty and not max_demand.empty:
            print("Applying yearly total and monthly maximum constraints.")
            def apply_constraints_func(forecast_df, total_demand, max_demand):
                df = forecast_df.copy()

                # Scale to yearly totals
                for year in total_demand['financial_year'].unique():
                    mask = df['financial_year'] == year
                    if mask.any():
                        current_sum = df.loc[mask, 'yhat'].sum()
                        target_sum = total_demand[total_demand['financial_year'] == year]['Total demand'].values[0]
                        # Avoid division by zero or scaling if current_sum is very small or zero
                        scale_factor = target_sum / current_sum if current_sum > 1e-9 else (target_sum if target_sum > 0 else 1) # Scale to target if current is near zero, else 1 if target is zero
                        df.loc[mask, 'yhat'] *= scale_factor
                        print(f"Scaled year {year} by factor {scale_factor:.4f}")


                # Adjust for monthly peaks
                for _, row in max_demand.iterrows():
                    year, month, target_max = row['financial_year'], row['month'], row['max_demand']
                    mask = (df['financial_year'] == year) & (df['month'] == month)
                    if mask.any():
                        current_max = df.loc[mask, 'yhat'].max()
                        # Only scale up if current max is below target max and target max is positive
                        if target_max > 0 and (current_max < target_max or current_max == 0):
                            scale_factor = target_max / current_max if current_max > 1e-9 else target_max # Scale to target if current max is near zero or zero
                            df.loc[mask, 'yhat'] *= scale_factor
                            print(f"Adjusted max for {year}-{month} by factor {scale_factor:.4f}")
                        # Optional: Add logic to scale down if current_max > target_max (if needed)

                return df[['ds', 'yhat']]
            final_forecast = apply_constraints_func(forecast_with_profiles, total_demand, max_demand)
        else:
            print("Constraints not applied.")
            final_forecast = forecast_with_profiles[['ds', 'yhat']].copy()

        # Ensure final forecast is non-negative
        final_forecast['yhat'] = final_forecast['yhat'].clip(lower=0)

        return final_forecast.round(2)

    except FileNotFoundError:
        print(f"Error: The file was not found at {excel_file_path}")
        return None
    except ValueError as ve:
        print(f"Error processing data: {ve}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        import traceback
        traceback.print_exc()
        return None

In [7]:
create_load_curve(excel_file_path,base_year)

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\holidays\countries\india.py:180: Warning: Requested Holidays are available only from 2001 to 2035.
  warnings.warn(warning_msg, Warning)
C:\Users\Admin\AppData\Local\Temp\ipykernel_8724\3530405515.py:231: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  future_dates = pd.date_range(start=forecast_start_date, end=forecast_end_date, freq='H')


Applying yearly total and monthly maximum constraints.
Scaled year 2026 by factor 1.0196
Scaled year 2027 by factor 1.0619
Scaled year 2028 by factor 1.1058
Scaled year 2029 by factor 1.1618
Scaled year 2030 by factor 1.2198
Scaled year 2031 by factor 1.2943
Scaled year 2032 by factor 1.3699
Scaled year 2033 by factor 1.4606
Scaled year 2034 by factor 1.5537
Scaled year 2035 by factor 1.6540
Scaled year 2036 by factor 1.7563
Scaled year 2037 by factor 1.8776
Adjusted max for 2026.0-4.0 by factor 1.0109
Adjusted max for 2028.0-4.0 by factor 1.0075
Adjusted max for 2029.0-4.0 by factor 1.0008
Adjusted max for 2026.0-5.0 by factor 1.1120
Adjusted max for 2027.0-5.0 by factor 1.0999
Adjusted max for 2028.0-5.0 by factor 1.1173
Adjusted max for 2029.0-5.0 by factor 1.1083
Adjusted max for 2030.0-5.0 by factor 1.1101
Adjusted max for 2031.0-5.0 by factor 1.0981
Adjusted max for 2032.0-5.0 by factor 1.0648
Adjusted max for 2033.0-5.0 by factor 1.0428
Adjusted max for 2034.0-5.0 by factor 1.03

,ds,yhat
0,2025-04-01 00:00:00,4825.30
1,2025-04-01 01:00:00,4547.88
2,2025-04-01 02:00:00,4323.21
3,2025-04-01 03:00:00,4127.32
4,2025-04-01 04:00:00,3993.03
...,...,...
113947,2038-03-31 19:00:00,5400.71
113948,2038-03-31 20:00:00,5416.92
113949,2038-03-31 21:00:00,5517.60
113950,2038-03-31 22:00:00,5669.36
